# NB07e — Stage 2: UNet++ + V6: SliceAttentionGate + TumorAwareHead (80/20) + 7-Slice + SmallBoost
**Experiment:** Architecture improvements over baseline NB04

**Changes from NB04:**
- ASPP module after CDCG fusion (multi-scale context)
- Deep Supervision at decoder layers (forces early feature learning)
- Stochastic ROI Perturbation augmentation (paste small tumors during training)

**Everything else identical:** same ROIs, same fold, same loss schedule

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from collections import OrderedDict
from scipy import ndimage
from scipy.ndimage import distance_transform_edt
import json, time, gc, warnings, math
warnings.filterwarnings('ignore')

import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

device = torch.device('cuda')
torch.backends.cudnn.benchmark = True

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.5.1+cu121
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


In [2]:
# ======== CONFIG ========
OUTPUT_DIR = Path(r"D:\fsdf_uca_kits\outputs")
ROI_GT_DIR = OUTPUT_DIR / "kidney_rois_gt"
STATS_DIR = OUTPUT_DIR / "stats"
CKPT_DIR = OUTPUT_DIR / "checkpoints_80_20"
FIG_DIR = OUTPUT_DIR / "figures" / "nb07e"

for d in [CKPT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 256
SLICE_CONTEXT = 7
NUM_CLASSES = 3
FUSE_CHANNELS = 48
ENCODER = 'efficientnet-b4'

BATCH_SIZE = 8
NUM_EPOCHS = 100  # Best ep50-65. May overfit after ep75.
BASE_LR = 3e-4
WEIGHT_DECAY = 1e-4
USE_FP16 = True
VAL_EVERY = 3
WARMUP_EPOCHS = 3
TUMOR_POS_RATIO = 0.70

BOUNDARY_START = 999  # disabled — using BW-EDL instead
BOUNDARY_MAX_WEIGHT = 0.5
BOUNDARY_RAMP = 10
EDL_START = 20
EDL_WEIGHT = 0.15

CURRENT_FOLD = 0
CACHE_SIZE = 30
EXP_NAME = 'nb07e'  # separate from nb04

print(f"NB07e: V6: SliceGate + TumorHead + BW-EDL")
print(f"  Batch: {BATCH_SIZE}, LR: {BASE_LR}, Epochs: {NUM_EPOCHS}")
print(f"  Fold: {CURRENT_FOLD}")

NB07e: V6: SliceGate + TumorHead + BW-EDL
  Batch: 8, LR: 0.0003, Epochs: 100
  Fold: 0


In [3]:
# ======== Load fold splits + ROI metadata ========
with open(STATS_DIR / 'fold_splits_80_20.json') as f:
    fold_data = json.load(f)

fold_key = f'fold_{CURRENT_FOLD}'
train_cases = fold_data['folds'][fold_key]['train']
val_cases = fold_data['folds'][fold_key]['val']

with open(ROI_GT_DIR / 'roi_meta_all.json') as f:
    all_roi_metas = json.load(f)

train_rois = [r for r in all_roi_metas if r['case_name'] in train_cases]
val_rois = [r for r in all_roi_metas if r['case_name'] in val_cases]

print(f"Train ROIs: {len(train_rois)} ({sum(1 for r in train_rois if r['has_tumor'])} with tumor)")
print(f"Val ROIs:   {len(val_rois)} ({sum(1 for r in val_rois if r['has_tumor'])} with tumor)")

Train ROIs: 328 (169 with tumor)
Val ROIs:   84 (44 with tumor)


## 1. Dataset with Stochastic ROI Perturbation

In [4]:
import torchvision.transforms.functional as TF
# ======== Dataset with Crop Augmentation + Aspect-Ratio Padding ========
import random

class KidneyTumorSliceDataset(torch.utils.data.Dataset):
    """V5 Dataset: Stochastic ROI Perturbation + Aspect-Ratio Preserving Resize.
    
    NOVELTY 1 - Simulated Crop Augmentation:
      Randomly shifts/scales crop boundaries during training.
      Simulates predicted ROI variation without needing Stage 1 predictions.
      Model learns to handle ANY crop style → robust to Stage 1 errors.
    
    NOVELTY 2 - Aspect-Ratio Preserving Resize:
      Pads shortest dimension to square BEFORE resize.
      Preserves physical anatomy shape.
    """
    def __init__(self, roi_metas, roi_dir, img_size=256, n_slices=7,
                 tumor_pos_ratio=0.5, augment=True, aug_degrees=15,
                 pred_roi_dir=None, pred_roi_metas=None, pred_mix_ratio=0.30,
                 crop_aug=True, crop_shift_max=15, crop_scale_range=(0.90, 1.10)):
        self.roi_dir = Path(roi_dir)
        self.img_size = img_size
        self.n_slices = n_slices
        self.augment = augment
        self.aug_degrees = aug_degrees
        self.crop_aug = crop_aug and augment  # only during training
        self.crop_shift_max = crop_shift_max
        self.crop_scale_range = crop_scale_range
        
        # Build slice index from GT ROIs
        gt_tumor_slices, gt_nontumor_slices = [], []
        for meta in roi_metas:
            roi_name = meta['roi_name']
            if not (self.roi_dir / f'{roi_name}_img.npy').exists(): continue
            D = meta['shape'][0]
            seg = np.load(self.roi_dir / f'{roi_name}_seg.npy', mmap_mode='r')
            for s in range(D):
                entry = (roi_name, s, 'gt')
                if (seg[s] == 2).any():
                    gt_tumor_slices.append(entry)
                else:
                    gt_nontumor_slices.append(entry)
        
        # Add predicted ROIs if available
        pred_tumor_slices = []
        if pred_roi_metas and pred_roi_dir:
            pred_roi_dir = Path(pred_roi_dir)
            for meta in (pred_roi_metas or []):
                roi_name = meta['roi_name']
                if not (pred_roi_dir / f'{roi_name}_img.npy').exists(): continue
                if not meta.get('has_tumor', False): continue
                D = meta['shape'][0]
                seg = np.load(pred_roi_dir / f'{roi_name}_seg.npy', mmap_mode='r')
                for s in range(D):
                    if (seg[s] == 2).any():
                        pred_tumor_slices.append((roi_name, s, 'pred'))
            
            # Enforce mix ratio
            if pred_tumor_slices:
                target_pred = int(len(gt_tumor_slices) * pred_mix_ratio / (1 - pred_mix_ratio))
                if len(pred_tumor_slices) > target_pred:
                    pred_tumor_slices = random.sample(pred_tumor_slices, target_pred)
                    print(f"  Pred tumor slices trimmed: → {len(pred_tumor_slices)} (target {pred_mix_ratio:.0%})")
        
        # Small tumor boost
        SMALL_TUMOR_THRESHOLD = 2731
        small_rois = set()
        for meta in roi_metas:
            if meta.get('has_tumor') and meta.get('n_tumor', 0) < SMALL_TUMOR_THRESHOLD:
                small_rois.add(meta['roi_name'])
        
        boosted_gt = []
        for entry in gt_tumor_slices:
            boosted_gt.append(entry)
            if entry[0] in small_rois:
                boosted_gt.extend([entry] * 2)  # 3x total
        
        all_tumor = boosted_gt + pred_tumor_slices
        n_nontumor = int(len(all_tumor) * (1 - tumor_pos_ratio) / tumor_pos_ratio)
        
        if len(gt_nontumor_slices) > n_nontumor:
            selected_nontumor = random.sample(gt_nontumor_slices, n_nontumor)
        else:
            selected_nontumor = gt_nontumor_slices
        
        self.slices = all_tumor + selected_nontumor
        random.shuffle(self.slices)
        
        gt_count = sum(1 for s in all_tumor if s[2] == 'gt')
        pred_count = sum(1 for s in all_tumor if s[2] == 'pred')
        small_boosted = sum(1 for s in boosted_gt if s[0] in small_rois) - len([s for s in gt_tumor_slices if s[0] in small_rois])
        
        print(f"  GT tumor slices: {gt_count}")
        print(f"  Pred tumor slices: {pred_count} (actual ratio: {100*pred_count/max(gt_count+pred_count,1):.1f}%)")
        print(f"  Small ROIs boosted 3x: {len(small_rois)}")
        print(f"  Total tumor slices: {len(all_tumor)}")
        print(f"  Non-tumor slices: {len(selected_nontumor)}")
        if self.crop_aug:
            print(f"  Crop augmentation: shift±{crop_shift_max}px, scale={crop_scale_range}")
        
        self.pred_roi_dir = Path(pred_roi_dir) if pred_roi_dir else None
    
    def _apply_crop_perturbation_square(self, img_sq, seg_sq, pad_val):
        """Stochastic ROI Perturbation on already-squared image.
        Shift + scale, then re-crop to original square size.
        CORRECT ORDER: called AFTER pad-to-square, BEFORE resize."""
        S = img_sq.shape[0]
        shift_max = max(1, int(S * 0.10))  # 10% of side length (relative, not fixed)
        dy = random.randint(-shift_max, shift_max)
        dx = random.randint(-shift_max, shift_max)
        scale = random.uniform(*self.crop_scale_range)
        new_S = int(S * scale)
        
        pad = max(abs(dy), abs(dx), abs(new_S - S)) + 5
        padded_img = np.pad(img_sq, pad, mode='constant', constant_values=pad_val)
        padded_seg = np.pad(seg_sq, pad, mode='constant', constant_values=0)
        
        cy, cx = pad + S // 2 + dy, pad + S // 2 + dx
        y1 = max(0, cy - new_S // 2)
        x1 = max(0, cx - new_S // 2)
        
        cropped_img = padded_img[y1:y1+new_S, x1:x1+new_S]
        cropped_seg = padded_seg[y1:y1+new_S, x1:x1+new_S]
        
        # Resize back to original square size S×S
        if cropped_img.shape[0] != S or cropped_img.shape[1] != S:
            cropped_img = np.array(F.interpolate(
                torch.from_numpy(cropped_img[None, None]).float(),
                (S, S), mode='bilinear', align_corners=False).squeeze())
            cropped_seg = np.array(F.interpolate(
                torch.from_numpy(cropped_seg[None, None].astype(np.float32)),
                (S, S), mode='nearest').squeeze()).astype(np.int64)
        
        return cropped_img, cropped_seg

    def _aspect_ratio_pad(self, img_2d, seg_2d, pad_val):
        """Pad to square preserving aspect ratio, THEN resize."""
        H, W = img_2d.shape
        if H == W:
            return img_2d, seg_2d
        
        max_dim = max(H, W)
        pad_h = (max_dim - H) // 2
        pad_w = (max_dim - W) // 2
        
        img_padded = np.pad(img_2d, ((pad_h, max_dim - H - pad_h), 
                                       (pad_w, max_dim - W - pad_w)),
                            mode='constant', constant_values=pad_val)
        seg_padded = np.pad(seg_2d, ((pad_h, max_dim - H - pad_h),
                                       (pad_w, max_dim - W - pad_w)),
                            mode='constant', constant_values=0)
        return img_padded, seg_padded
    
    def __len__(self):
        return len(self.slices)
    
    def __getitem__(self, idx):
        roi_name, slice_idx, source = self.slices[idx]
        roi_dir = self.pred_roi_dir if source == 'pred' and self.pred_roi_dir else self.roi_dir
        
        img = np.load(roi_dir / f'{roi_name}_img.npy', mmap_mode='r')
        seg = np.load(roi_dir / f'{roi_name}_seg.npy', mmap_mode='r')
        D, H, W = img.shape
        half = self.n_slices // 2
        
        # Extract multi-slice context
        slices = np.zeros((self.n_slices, H, W), dtype=np.float32)
        for j in range(self.n_slices):
            si = max(0, min(D - 1, slice_idx - half + j))
            slices[j] = img[si]
        seg_slice = seg[slice_idx].copy().astype(np.int64)
        
        # === CORRECT ORDER: 1) Pad to square → 2) Crop perturbation → 3) Resize ===
        
        # Step 1: Pad to square (preserves aspect ratio)
        pad_val = float(slices.min())
        max_dim = max(H, W)
        sq_slices = np.zeros((self.n_slices, max_dim, max_dim), dtype=np.float32)
        for j in range(self.n_slices):
            sq_slices[j], _ = self._aspect_ratio_pad(slices[j], seg_slice, pad_val)
        _, seg_sq = self._aspect_ratio_pad(slices[self.n_slices // 2], seg_slice, 0)
        
        # Step 2: Stochastic crop perturbation (GT ROIs only, p=0.4)
        if self.crop_aug and random.random() < 0.3:  # p=0.3 on all predicted ROIs (NB07e: no GT ROIs)
            aug_sq = np.zeros_like(sq_slices)
            # Same perturbation for all slices in stack
            for j in range(self.n_slices):
                aug_sq[j], _ = self._apply_crop_perturbation_square(sq_slices[j], seg_sq, pad_val)
            _, seg_sq = self._apply_crop_perturbation_square(
                sq_slices[self.n_slices // 2], seg_sq, pad_val)
            sq_slices = aug_sq
        
        # Step 3: Resize to target 256×256
        slices_t = F.interpolate(
            torch.from_numpy(sq_slices[None]).float(),
            (self.img_size, self.img_size), mode='bilinear', align_corners=False
        ).squeeze(0)
        seg_t = F.interpolate(
            torch.from_numpy(seg_sq[None, None].astype(np.float32)),
            (self.img_size, self.img_size), mode='nearest'
        ).squeeze().long()
        
        # Standard augmentations (flips, rotation)
        if self.augment:
            if random.random() < 0.5:
                slices_t = slices_t.flip(-1)
                seg_t = seg_t.flip(-1)
            if random.random() < 0.5:
                slices_t = slices_t.flip(-2)
                seg_t = seg_t.flip(-2)
            if self.aug_degrees > 0 and random.random() < 0.3:
                angle = random.uniform(-self.aug_degrees, self.aug_degrees)
                slices_t = TF.rotate(slices_t, angle, interpolation=TF.InterpolationMode.BILINEAR)
                seg_t = TF.rotate(seg_t.unsqueeze(0).float(), angle, interpolation=TF.InterpolationMode.NEAREST).squeeze().long()
        
        return slices_t, seg_t

    def shuffle_epoch(self):
        random.shuffle(self.slices)

print("V5a Dataset ready: pad→crop→resize order, relative perturbation, GT-only aug")


V5a Dataset ready: pad→crop→resize order, relative perturbation, GT-only aug


In [5]:
# ======== Build Datasets: 100% nnU-Net Predicted ROIs ========
# KEY CHANGE from NB07e: Both train AND val use predicted ROIs.
# This eliminates train/test distribution mismatch entirely.
# Matches standard cascade methodology (what other papers do).

import json
from pathlib import Path

ROI_GT_DIR = OUTPUT_DIR / "kidney_rois_gt"
ROI_PRED_DIR = OUTPUT_DIR / "kidney_rois_nnunet_80_20"

# Load predicted ROI metadata
with open(ROI_PRED_DIR / "roi_meta_all.json") as f:
    pred_metas = json.load(f)

# Split into train/val based on fold
train_pred_metas = [m for m in pred_metas if m['case_name'] in set(train_cases)]
val_pred_metas = [m for m in pred_metas if m['case_name'] in set(val_cases)]

# Safety check
if len(train_pred_metas) == 0:
    raise ValueError(
        "Zero predicted ROIs for training cases. "
        "Run NB06 Cell 8 (ROI extraction) on ALL 208 cases first."
    )
if len(val_pred_metas) == 0:
    raise ValueError(
        "Zero predicted ROIs for validation cases. "
        "Run NB06 Cell 8 (ROI extraction) on ALL 208 cases first."
    )

print(f"100% Predicted ROI Training")
print(f"  Train predicted ROIs: {len(train_pred_metas)}")
print(f"  Val predicted ROIs:   {len(val_pred_metas)}")
print(f"  Train tumor ROIs: {sum(1 for m in train_pred_metas if m['has_tumor'])}")
print(f"  Val tumor ROIs:   {sum(1 for m in val_pred_metas if m['has_tumor'])}")

print(f"\nBuilding datasets...")

# TRAIN: 100% predicted ROIs (NO GT ROIs, NO mixing)
# Crop aug still applied for generalization (p=0.3)
train_ds = KidneyTumorSliceDataset(
    roi_metas=train_pred_metas,       # predicted ROIs as primary
    roi_dir=ROI_PRED_DIR,             # predicted ROI directory
    img_size=IMG_SIZE, n_slices=SLICE_CONTEXT,
    tumor_pos_ratio=0.5, augment=True, aug_degrees=15,
    pred_roi_dir=None,                # NO mixing — all predicted
    pred_roi_metas=None,
    pred_mix_ratio=0.0,               # 0% mix — pure predicted
    crop_aug=True,                    # still useful for generalization
    crop_shift_max=15,
    crop_scale_range=(0.90, 1.10),
)

# VAL: 100% predicted ROIs (matches test distribution!)
# This is the KEY difference — val now matches deployment
val_ds = KidneyTumorSliceDataset(
    roi_metas=val_pred_metas,         # predicted ROIs
    roi_dir=ROI_PRED_DIR,             # predicted ROI directory
    img_size=IMG_SIZE, n_slices=SLICE_CONTEXT,
    tumor_pos_ratio=0.5, augment=False,
    crop_aug=False,                   # no aug for validation
)

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=0, pin_memory=True, drop_last=True)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"\nTrain: {len(train_ds)} slices, {len(train_loader)} batches")
print(f"Val:   {len(val_ds)} slices")

# Verify batch shape
batch_img, batch_seg = next(iter(train_loader))
print(f"Batch: img={batch_img.shape}, seg={batch_seg.shape}, unique={batch_seg.unique().tolist()}")


100% Predicted ROI Training
  Train predicted ROIs: 405
  Val predicted ROIs:   99
  Train tumor ROIs: 169
  Val tumor ROIs:   44

Building datasets...
  GT tumor slices: 3067
  Pred tumor slices: 0 (actual ratio: 0.0%)
  Small ROIs boosted 3x: 26
  Total tumor slices: 3067
  Non-tumor slices: 3067
  Crop augmentation: shift±15px, scale=(0.9, 1.1)
  GT tumor slices: 738
  Pred tumor slices: 0 (actual ratio: 0.0%)
  Small ROIs boosted 3x: 4
  Total tumor slices: 738
  Non-tumor slices: 738

Train: 6134 slices, 766 batches
Val:   1476 slices
Batch: img=torch.Size([8, 7, 256, 256]), seg=torch.Size([8, 256, 256]), unique=[0, 1, 2]


## 2. Architecture with ASPP + Deep Supervision

In [6]:
# ======== V6 Architecture: V5 + SliceAttentionGate + TumorAwareHead ========

class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels, rates=[1, 6, 12, 18]):
        super().__init__()
        self.convs = nn.ModuleList()
        for rate in rates:
            self.convs.append(nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=rate, dilation=rate),
                nn.InstanceNorm2d(out_channels), nn.ReLU(inplace=True)))
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_channels, out_channels, 1), nn.ReLU(True))
        self.fuse = nn.Sequential(
            nn.Conv2d(out_channels * (len(rates) + 1), out_channels, 1),
            nn.InstanceNorm2d(out_channels), nn.ReLU(True))
    def forward(self, x):
        feats = [conv(x) for conv in self.convs]
        feats.append(F.interpolate(self.gap(x), size=x.shape[2:], mode='bilinear', align_corners=False))
        return self.fuse(torch.cat(feats, dim=1))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        r = channels // reduction
        self.ca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(channels, r), nn.ReLU(True), nn.Linear(r, channels))
        self.ca_max = nn.Sequential(nn.AdaptiveMaxPool2d(1), nn.Flatten(), nn.Linear(channels, r), nn.ReLU(True), nn.Linear(r, channels))
        self.sa = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3), nn.Sigmoid())
        self.spatial_attn = None
    def forward(self, x):
        x = x * torch.sigmoid(self.ca(x) + self.ca_max(x)).unsqueeze(-1).unsqueeze(-1)
        sa = self.sa(torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], 1))
        self.spatial_attn = sa.detach()
        return x * sa


class SliceAttentionGate(nn.Module):
    """CSAM-inspired input-level slice attention for 7-slice 2.5D CT input.
    NOT a faithful CSAM implementation - adapted for input-level application.
    Ref: Hung et al., CSAM, WACV 2024.
    
    Spatial gate: WHERE to attend [0.5, 1.5], init ~1.0
    Slice gate:   WHICH slices matter (softmax competition), init ~1.0
    Zero-init ensures identity-like start.
    """
    def __init__(self, num_slices=7, pos_kernel=7):
        super().__init__()
        self.num_slices = num_slices
        pad = pos_kernel // 2
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=pos_kernel, padding=pad, bias=True)
        self.slice_conv = nn.Conv2d(num_slices, num_slices, kernel_size=1, bias=True)
        nn.init.zeros_(self.spatial_conv.weight)
        nn.init.zeros_(self.spatial_conv.bias)
        nn.init.zeros_(self.slice_conv.weight)
        nn.init.zeros_(self.slice_conv.bias)

    def forward(self, x):
        max_pool = x.max(dim=1, keepdim=True)[0]
        avg_pool = x.mean(dim=1, keepdim=True)
        spatial_logits = self.spatial_conv(torch.cat([max_pool, avg_pool], dim=1))
        spatial_gate = 0.5 + torch.sigmoid(spatial_logits)
        x = x * spatial_gate
        slice_logits = self.slice_conv(x)
        slice_gate = torch.softmax(slice_logits, dim=1) * self.num_slices
        x = x * slice_gate
        return x


class TumorAwareHead(nn.Module):
    """Coarse tumor detection from bottleneck. Returns LOGITS."""
    def __init__(self, in_channels):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(in_channels, 64, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1),
        )
    def forward(self, bottleneck):
        return self.head(bottleneck)


class FSDFUCANetV6(nn.Module):
    """V6: V5 + SliceAttentionGate + TumorAwareHead
    SliceGate -> UNet++ -> ASPP -> CBAM -> SegHead + EDL + TumorHead"""
    def __init__(self, in_channels=7, num_classes=3, fuse_channels=48,
                 encoder_name='efficientnet-b4'):
        super().__init__()
        self.slice_gate = SliceAttentionGate(num_slices=in_channels)
        self.spatial_unet = smp.UnetPlusPlus(
            encoder_name=encoder_name, encoder_weights='imagenet',
            in_channels=in_channels, classes=fuse_channels)
        self.aspp = ASPP(fuse_channels, fuse_channels, rates=[1, 6, 12, 18])
        self.cbam = CBAM(channels=fuse_channels, reduction=8)
        self.seg_head = nn.Sequential(
            nn.Conv2d(fuse_channels, fuse_channels, 3, padding=1),
            nn.InstanceNorm2d(fuse_channels), nn.ReLU(True),
            nn.Conv2d(fuse_channels, num_classes, 1))
        self.ds_head = nn.Conv2d(fuse_channels, num_classes, 1)
        self.edl_head = nn.Sequential(
            nn.Conv2d(fuse_channels, fuse_channels, 3, padding=1),
            nn.InstanceNorm2d(fuse_channels), nn.ReLU(True),
            nn.Conv2d(fuse_channels, num_classes, 1),
            nn.Softplus())
        # Find bottleneck channels dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, 64, 64)
            feats = self.spatial_unet.encoder(dummy)
            bottleneck_ch = feats[-1].shape[1]
        self.tumor_head = TumorAwareHead(in_channels=bottleneck_ch)
        print(f"  TumorAwareHead bottleneck: {bottleneck_ch} channels")

    def forward(self, x, return_uncertainty=False, return_ds=False):
        x = self.slice_gate(x)
        encoder_features = self.spatial_unet.encoder(x)
        bottleneck = encoder_features[-1]
        tumor_logits = self.tumor_head(bottleneck)
        decoder_out = self.spatial_unet.decoder(encoder_features)
        spatial_feat = self.spatial_unet.segmentation_head(decoder_out)
        fused = self.aspp(spatial_feat)
        refined = self.cbam(fused)
        logits = self.seg_head(refined)
        out = {"logits": logits, "tumor_logits": tumor_logits}
        if return_ds:
            out['ds_logits'] = self.ds_head(fused)
        if return_uncertainty:
            evidence = self.edl_head(refined)
            alpha = evidence + 1.0
            uncertainty = NUM_CLASSES / alpha.sum(dim=1, keepdim=True)
            out['evidence'] = evidence
            out['uncertainty'] = uncertainty
        return out

_m = FSDFUCANetV6(SLICE_CONTEXT, NUM_CLASSES, FUSE_CHANNELS, ENCODER)
total_p = sum(p.numel() for p in _m.parameters())
gate_p = sum(p.numel() for p in _m.slice_gate.parameters())
tumor_p = sum(p.numel() for p in _m.tumor_head.parameters())
print(f"FSDFUCANetV6: {total_p:,} params")
print(f"  SliceAttentionGate: {gate_p:,} params")
print(f"  TumorAwareHead: {tumor_p:,} params")
print(f"  Base (V5): {total_p - gate_p - tumor_p:,} params")
del _m


  TumorAwareHead bottleneck: 448 channels
FSDFUCANetV6: 20,991,036 params
  SliceAttentionGate: 155 params
  TumorAwareHead: 28,801 params
  Base (V5): 20,962,080 params


In [7]:
model = FSDFUCANetV6(
    in_channels=SLICE_CONTEXT, num_classes=NUM_CLASSES,
    fuse_channels=FUSE_CHANNELS, encoder_name=ENCODER,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"FSDFUCANet-V6: {total_params:,} params")

with torch.no_grad():
    test = torch.randn(BATCH_SIZE, SLICE_CONTEXT, IMG_SIZE, IMG_SIZE, device=device)
    with torch.amp.autocast('cuda'):
        out = model(test)
        out_ds = model(test, return_ds=True)
    vram = torch.cuda.max_memory_allocated() / 1e9
    print(f"VRAM (batch={BATCH_SIZE}): {vram:.2f} GB")
    print(f"Output: {out['logits'].shape}")
    if isinstance(out_ds, dict):
        print(f"DS output: {out_ds['logits'].shape}, {out_ds['ds_logits'].shape}")
    del test, out, out_ds
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print("Model OK.")

  TumorAwareHead bottleneck: 448 channels
FSDFUCANet-V6: 20,991,036 params
VRAM (batch=8): 1.65 GB
Output: torch.Size([8, 3, 256, 256])
DS output: torch.Size([8, 3, 256, 256]), torch.Size([8, 3, 256, 256])
Model OK.


## 3. Loss + Optimizer

In [8]:
# ======== Losses: DiceFocal + BW-EDL +  + DS ========
from monai.losses import DiceFocalLoss
from scipy.ndimage import distance_transform_edt

# Standard DiceFocal
try:
    dice_focal = DiceFocalLoss(to_onehot_y=True, softmax=True, include_background=False,
                                weight=torch.tensor([0.3, 0.7]).to(device))
    _t = dice_focal(torch.randn(2,3,32,32,device=device), torch.randint(0,3,(2,1,32,32),device=device))
    print(f"DiceFocal OK: {_t.item():.4f}")
except:
    dice_focal = DiceFocalLoss(to_onehot_y=True, softmax=True, include_background=True,
                                weight=torch.tensor([0.1, 0.3, 0.6]).to(device))
    print("Fallback 3-weight DiceFocal")

# Boundary-Weighted EDL (NOVEL)
class BoundaryWeightedEDL(nn.Module):
    """NOVEL: Spatially-weighted evidential loss.
    High penalty at tumor CORE (must be confident).
    Low penalty at tumor BOUNDARY (uncertainty allowed).
    
    This changes the optimization landscape — cannot be ignored
    by the network unlike auxiliary branches."""
    def __init__(self, num_classes=3, core_weight=3.0, border_width=5):
        super().__init__()
        self.num_classes = num_classes
        self.core_weight = core_weight
        self.border_width = border_width
    
    def _get_boundary_weight(self, seg_np):
        """Compute distance-based weight map.
        0 at exact border, 1 at core/far from tumor."""
        tumor_mask = (seg_np == 2).astype(np.uint8)
        if tumor_mask.sum() == 0:
            return np.ones_like(seg_np, dtype=np.float32)
        
        # Distance from tumor border
        dist_inside = distance_transform_edt(tumor_mask)
        dist_outside = distance_transform_edt(1 - tumor_mask)
        
        # Near border: low weight. At core: high weight.
        border_dist = np.minimum(dist_inside, dist_outside)
        weight = np.clip(border_dist / self.border_width, 0, 1)
        
        # Scale: border=0.5 (allow uncertainty), core=core_weight (must be confident)
        weight = 0.5 + weight * (self.core_weight - 0.5)
        return weight.astype(np.float32)
    
    def forward(self, evidence, seg):
        alpha = evidence + 1.0
        S = alpha.sum(dim=1, keepdim=True)
        seg_oh = F.one_hot(seg, self.num_classes).permute(0, 3, 1, 2).float()
        
        # Standard EDL loss per pixel
        edl = (seg_oh * (torch.digamma(S) - torch.digamma(alpha))).sum(dim=1)
        
        # Compute boundary weight map (no cache — crop aug makes each sample unique)
        weights = []
        for b in range(seg.shape[0]):
            weights.append(self._get_boundary_weight(seg[b].cpu().numpy()))
        
        weight_map = torch.from_numpy(np.stack(weights)).to(evidence.device)
        
        return (edl * weight_map).mean()

# Legacy boundary loss (kept but disabled via BOUNDARY_START=999)
class BoundaryLoss(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.num_classes = num_classes
        self._cache = {}
    def _sdt(self, seg_np):
        H, W = seg_np.shape
        sdt = np.zeros((self.num_classes, H, W), dtype=np.float32)
        for c in range(self.num_classes):
            m = (seg_np == c).astype(np.uint8)
            if 0 < m.sum() < m.size:
                sdt[c] = distance_transform_edt(1-m) - distance_transform_edt(m)
        return sdt
    def forward(self, pred_soft, seg):
        sdts = []
        for b in range(seg.shape[0]):
            key = hash(seg[b].cpu().numpy().tobytes())
            if key not in self._cache:
                self._cache[key] = self._sdt(seg[b].cpu().numpy())
                if len(self._cache) > 3000:
                    for k in list(self._cache.keys())[:1500]: del self._cache[k]
            sdts.append(self._cache[key])
        return (pred_soft * torch.from_numpy(np.stack(sdts)).to(pred_soft.device)).mean()

# Legacy EDL (replaced by BW-EDL but kept for compatibility)
class EDLLoss(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.num_classes = num_classes
    def forward(self, evidence, seg):
        alpha = evidence + 1.0
        S = alpha.sum(dim=1, keepdim=True)
        seg_oh = F.one_hot(seg, self.num_classes).permute(0, 3, 1, 2).float()
        return (seg_oh * (torch.digamma(S) - torch.digamma(alpha))).sum(dim=1).mean()

boundary_loss = BoundaryLoss(NUM_CLASSES)
edl_loss = BoundaryWeightedEDL(NUM_CLASSES, core_weight=3.0, border_width=5)
ds_weight = 0.3

print(f"Losses ready:")
print(f"  DiceFocal (primary segmentation)")
print(f"  BoundaryWeightedEDL (NOVEL: spatially-calibrated uncertainty)")
print(f"  DeepSup (auxiliary)")
print(f"  Old BoundaryLoss (disabled, BOUNDARY_START=999)")

tumor_bce = nn.BCEWithLogitsLoss()
print("Tumor BCE loss defined")


DiceFocal OK: 0.5089
Losses ready:
  DiceFocal (primary segmentation)
  BoundaryWeightedEDL (NOVEL: spatially-calibrated uncertainty)
  DeepSup (auxiliary)
  Old BoundaryLoss (disabled, BOUNDARY_START=999)
Tumor BCE loss defined


In [9]:
# ======== Optimizer + Scheduler ========
optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1/WARMUP_EPOCHS, total_iters=WARMUP_EPOCHS)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS, eta_min=1e-6)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])
scaler = torch.amp.GradScaler('cuda') if USE_FP16 else None
print(f"Optimizer: AdamW LR={BASE_LR}, Warmup {WARMUP_EPOCHS}ep + Cosine")


Optimizer: AdamW LR=0.0003, Warmup 3ep + Cosine


## 4. Training Loop

In [10]:
print("=" * 70)
print(f"NB07e: FSDF-UCA-Net V6 (V6: SliceGate + TumorHead + BW-EDL)")
tumor_bce = nn.BCEWithLogitsLoss()
print(f"{total_params:,} params | Batch {BATCH_SIZE} | Fold {CURRENT_FOLD}")
print("=" * 70)

history = {'loss':[], 'loss_seg':[], 'loss_bnd':[], 'loss_edl':[], 'loss_ds':[],
           'kidney_dice':[], 'tumor_dice':[], 'lr':[]}
best_tumor_dice = 0.0
best_epoch = 0
total_start = time.time()

for epoch in range(NUM_EPOCHS):
    ep_start = time.time()
    model.train()
    train_ds.shuffle_epoch()

    ep_loss, ep_seg, ep_bnd, ep_edl, ep_ds = 0, 0, 0, 0, 0
    n_batches = 0
    nan_count = 0

    bnd_w = min(BOUNDARY_MAX_WEIGHT, BOUNDARY_MAX_WEIGHT * (epoch - BOUNDARY_START) / BOUNDARY_RAMP) if epoch >= BOUNDARY_START else 0
    # EDL ramp: 0 before EDL_START, linear ramp over 10 epochs
    edl_ramp = min(1.0, max(0.0, (epoch - EDL_START) / 10.0)) if epoch >= EDL_START else 0.0
    use_edl = edl_ramp > 0

    for batch_img, batch_seg in train_loader:
        batch_img = batch_img.to(device, non_blocking=True)
        batch_seg = batch_seg.to(device, non_blocking=True)

        ctx = torch.amp.autocast('cuda') if USE_FP16 else torch.amp.autocast('cuda', enabled=False)
        with ctx:
            if use_edl:
                out = model(batch_img, return_uncertainty=True, return_ds=True)
                logits = out['logits']
                ds_logits = out['ds_logits']
                tumor_logits = out['tumor_logits']
                evidence = out['evidence']
            else:
                out = model(batch_img, return_ds=True)
                logits = out['logits']
                ds_logits = out['ds_logits']
                tumor_logits = out['tumor_logits']

            seg_for_loss = batch_seg.unsqueeze(1)
            l_seg = dice_focal(logits, seg_for_loss)
            l_ds = dice_focal(ds_logits, seg_for_loss) * ds_weight
            l_bnd = boundary_loss(F.softmax(logits, dim=1), batch_seg) if bnd_w > 0 else torch.tensor(0.0, device=device)
            l_edl = edl_loss(evidence, batch_seg) if use_edl else torch.tensor(0.0, device=device)

            # Tumor-aware auxiliary loss
            tumor_target = (batch_seg == 2).float().unsqueeze(1)
            tumor_target_ds = F.interpolate(tumor_target, size=tumor_logits.shape[2:], mode="nearest")
            l_tum = tumor_bce(tumor_logits, tumor_target_ds)
            loss = l_seg + l_ds + bnd_w * l_bnd + EDL_WEIGHT * edl_ramp * l_edl
            loss = loss + 0.1 * l_tum

        if torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            if scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            ep_loss += loss.item(); ep_seg += l_seg.item()
            ep_bnd += l_bnd.item(); ep_edl += l_edl.item()
            ep_ds += l_ds.item(); n_batches += 1
        else:
            nan_count += 1

        if use_edl:
            del batch_img, batch_seg, logits, ds_logits, evidence, tumor_logits
        else:
            del batch_img, batch_seg, logits, ds_logits, tumor_logits

    scheduler.step()
    # RAM cleanup
    if (epoch + 1) % 5 == 0:
        gc.collect()
        if len(boundary_loss._cache) > 2000:
            boundary_loss._cache.clear()
    al, asg, ab, ae, ad = [x/max(n_batches,1) for x in [ep_loss, ep_seg, ep_bnd, ep_edl, ep_ds]]
    lr = optimizer.param_groups[0]['lr']
    for k, v in zip(['loss','loss_seg','loss_bnd','loss_edl','loss_ds','lr'], [al,asg,ab,ae,ad,lr]):
        history[k].append(v)

    if nan_count: print(f"    WARNING: {nan_count} NaN")

    do_val = (epoch+1) % VAL_EVERY == 0 or epoch == 0 or epoch == NUM_EPOCHS-1
    if do_val:
        torch.cuda.empty_cache()
        model.eval()
        tp = {c:0 for c in range(NUM_CLASSES)}; fp = {c:0 for c in range(NUM_CLASSES)}; fn = {c:0 for c in range(NUM_CLASSES)}
        with torch.no_grad():
            for bi, bs in val_loader:
                bi, bs = bi.to(device), bs.to(device)
                pred = model(bi)["logits"].argmax(1)
                for c in range(NUM_CLASSES):
                    tp[c]+=(((pred==c)&(bs==c)).sum().item())
                    fp[c]+=(((pred==c)&(bs!=c)).sum().item())
                    fn[c]+=(((pred!=c)&(bs==c)).sum().item())
        kd = 2*tp[1]/max(2*tp[1]+fp[1]+fn[1],1)
        td = 2*tp[2]/max(2*tp[2]+fp[2]+fn[2],1)

        if td > best_tumor_dice:
            best_tumor_dice = td; best_epoch = epoch+1
            torch.save({'epoch':epoch+1, 'model_state':model.state_dict(),
                        'tumor_dice':td, 'kidney_dice':kd, 'fold':CURRENT_FOLD},
                       CKPT_DIR / f'{EXP_NAME}_fold{CURRENT_FOLD}_best.pth')

        # Save checkpoints at key epochs for E2E comparison
        if (epoch + 1) in [6, 9, 15, 30, 45, 60, 75, 100]:
            ep_ckpt = CKPT_DIR / f'{EXP_NAME}_fold{CURRENT_FOLD}_ep{epoch+1}.pth'
            torch.save({'epoch': epoch+1, 'model_state': model.state_dict(),
                        'tumor_dice': td, 'kidney_dice': kd, 'fold': CURRENT_FOLD}, ep_ckpt)

        history['kidney_dice'].append(kd); history['tumor_dice'].append(td)
        flag = ' << BEST' if td >= best_tumor_dice - 0.001 else ''
        val_str = f'KidD:{kd:.4f} TumD:{td:.4f}{flag}'
    else:
        history['kidney_dice'].append(history['kidney_dice'][-1] if history['kidney_dice'] else 0)
        history['tumor_dice'].append(history['tumor_dice'][-1] if history['tumor_dice'] else 0)
        val_str = '(no val)'

    loss_str = f'Seg:{asg:.4f} DS:{ad:.4f}'
    if bnd_w > 0: loss_str += f' Bnd:{ab:.4f}(w={bnd_w:.2f})'
    if use_edl: loss_str += f' EDL:{ae:.4f}(r={edl_ramp:.2f})'

    et = time.time() - ep_start
    print(f"  Ep{epoch+1:3d}/{NUM_EPOCHS} {loss_str} | {val_str} | LR:{lr:.2e} {et:.0f}s [{(time.time()-total_start)/3600:.1f}h]")

torch.save({'epoch':NUM_EPOCHS, 'model_state':model.state_dict(),
            'tumor_dice':history['tumor_dice'][-1], 'kidney_dice':history['kidney_dice'][-1],
            'fold':CURRENT_FOLD}, CKPT_DIR / f'{EXP_NAME}_fold{CURRENT_FOLD}_final.pth')

print(f"\nDone in {(time.time()-total_start)/60:.1f} min")
print(f"Best Tumor Dice: {best_tumor_dice:.4f} at epoch {best_epoch}")

NB07e: FSDF-UCA-Net V6 (V6: SliceGate + TumorHead + BW-EDL)
20,991,036 params | Batch 8 | Fold 0
  Ep  1/100 Seg:0.4052 DS:0.1204 | KidD:0.8544 TumD:0.7292 << BEST | LR:1.67e-04 418s [0.1h]
  Ep  2/100 Seg:0.3318 DS:0.1009 | (no val) | LR:2.33e-04 367s [0.2h]
  Ep  3/100 Seg:0.3118 DS:0.0946 | KidD:0.9101 TumD:0.8317 << BEST | LR:3.00e-04 397s [0.3h]
  Ep  4/100 Seg:0.3035 DS:0.0918 | (no val) | LR:3.00e-04 366s [0.4h]
  Ep  5/100 Seg:0.2969 DS:0.0897 | (no val) | LR:3.00e-04 365s [0.5h]
  Ep  6/100 Seg:0.2921 DS:0.0882 | KidD:0.9053 TumD:0.8049 | LR:2.99e-04 391s [0.6h]
  Ep  7/100 Seg:0.2915 DS:0.0880 | (no val) | LR:2.99e-04 372s [0.7h]
  Ep  8/100 Seg:0.2867 DS:0.0865 | (no val) | LR:2.98e-04 375s [0.8h]
  Ep  9/100 Seg:0.2867 DS:0.0864 | KidD:0.9295 TumD:0.8534 << BEST | LR:2.97e-04 409s [1.0h]
  Ep 10/100 Seg:0.2831 DS:0.0854 | (no val) | LR:2.96e-04 363s [1.1h]
  Ep 11/100 Seg:0.2821 DS:0.0851 | (no val) | LR:2.95e-04 361s [1.2h]
  Ep 12/100 Seg:0.2820 DS:0.0850 | KidD:0.9185 Tu

In [11]:
# ======== Summary ========
print(f"\n{'='*70}")
print(f"NB07e SUMMARY — V6: SliceGate + TumorHead + BW-EDL (Fold {CURRENT_FOLD})")
print(f"{'='*70}")
print(f"  Params: {total_params:,}")
print(f"  Training: {NUM_EPOCHS} epochs, {(time.time()-total_start)/3600:.1f} hours")
print(f"  Best Tumor Dice (slice): {best_tumor_dice:.4f} (epoch {best_epoch})")
print(f"  Checkpoints: {EXP_NAME}_fold{CURRENT_FOLD}_best.pth, {EXP_NAME}_fold{CURRENT_FOLD}_final.pth")
print(f"\n  Compare with NB04 baseline: NB04=0.9199, NB04c=0.9208 (slice)")
print(f"  This experiment:            TumDice={best_tumor_dice:.4f} (slice)")
print(f"  Difference:                 {best_tumor_dice - 0.9199:+.4f}")


NB07e SUMMARY — V6: SliceGate + TumorHead + BW-EDL (Fold 0)
  Params: 20,991,036
  Training: 100 epochs, 10.6 hours
  Best Tumor Dice (slice): 0.8820 (epoch 60)
  Checkpoints: nb07e_fold0_best.pth, nb07e_fold0_final.pth

  Compare with NB04 baseline: NB04=0.9199, NB04c=0.9208 (slice)
  This experiment:            TumDice=0.8820 (slice)
  Difference:                 -0.0379


In [12]:
# ======== E2E Evaluation: NB07e ep60 on nnU-Net ROIs ========
import numpy as np
import json, time
from pathlib import Path
from scipy.ndimage import gaussian_filter

FOLD_PATH = Path(r"D:\fsdf_uca_kits\outputs\stats\fold_splits_80_20.json")
ROI_NNUNET_DIR = OUTPUT_DIR / "kidney_rois_nnunet_80_20"

model = FSDFUCANetV6(SLICE_CONTEXT, NUM_CLASSES, FUSE_CHANNELS, ENCODER).to(device)
model_edl = FSDFUCANetV6(SLICE_CONTEXT, NUM_CLASSES, FUSE_CHANNELS, ENCODER).to(device)

# Load BEST checkpoint (ep60)
ckpt = torch.load(CKPT_DIR / 'nb07e_fold0_best.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
model_edl.load_state_dict(ckpt['model_state'])
model_edl.eval()
print(f"Loaded: epoch {ckpt['epoch']}, TumD={ckpt['tumor_dice']:.4f}")

with open(FOLD_PATH) as f:
    splits = json.load(f)
val_case_names = set(splits['folds']['fold_0']['val'])

with open(ROI_NNUNET_DIR / "roi_meta_all.json") as f:
    nn_metas = json.load(f)
nn_val = [m for m in nn_metas if m['case_name'] in val_case_names]
cases_tumor = sorted(set(m['case_name'] for m in nn_val if m['has_tumor']))
print(f"Val cases with tumor: {len(cases_tumor)}")

def infer_roi_v5(model, roi_name, roi_dir, img_size=256, n_slices=7, device='cuda'):
    img = np.load(roi_dir / f'{roi_name}_img.npy')
    seg = np.load(roi_dir / f'{roi_name}_seg.npy')
    D, H, W = img.shape
    half = n_slices // 2
    prob_vol = np.zeros((3, D, H, W), dtype=np.float32)
    
    model.eval()
    with torch.no_grad():
        for s in range(D):
            slices = np.zeros((n_slices, H, W), dtype=np.float32)
            for j in range(n_slices):
                si = max(0, min(D-1, s - half + j))
                slices[j] = img[si]
            
            # Aspect-ratio pad to square (match training)
            max_dim = max(H, W)
            pad_h = (max_dim - H) // 2
            pad_w = (max_dim - W) // 2
            pad_val = float(slices.min())
            sq = np.full((n_slices, max_dim, max_dim), pad_val, dtype=np.float32)
            sq[:, pad_h:pad_h+H, pad_w:pad_w+W] = slices
            
            inp = torch.from_numpy(sq[None]).float().to(device)
            inp = torch.nn.functional.interpolate(inp, (img_size, img_size),
                                                    mode='bilinear', align_corners=False)
            logits = model(inp)["logits"]  # V6 always returns tuple
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            
            # Resize back to square then unpad
            probs_full = np.zeros((3, max_dim, max_dim), dtype=np.float32)
            for c in range(3):
                probs_full[c] = torch.nn.functional.interpolate(
                    torch.from_numpy(probs[c:c+1, None]),
                    (max_dim, max_dim), mode='bilinear', align_corners=False
                ).numpy()[0, 0]
            prob_vol[:, s] = probs_full[:, pad_h:pad_h+H, pad_w:pad_w+W]
    
    for c in range(3):
        prob_vol[c] = gaussian_filter(prob_vol[c], sigma=[1.0, 0.5, 0.5])
    pred_vol = prob_vol.argmax(axis=0).astype(np.uint8)
    return pred_vol, seg

results = []
t0 = time.time()

for ci, case_name in enumerate(cases_tumor):
    pred_rois = sorted(ROI_NNUNET_DIR.glob(f"{case_name}_roi*_img.npy"))
    tp_m, fp_m, fn_m, tp_t, fp_t, fn_t = 0,0,0,0,0,0
    
    for p in pred_rois:
        roi_name = p.stem.replace('_img', '')
        if not (ROI_NNUNET_DIR / f'{roi_name}_seg.npy').exists(): continue
        pred_vol, gt_vol = infer_roi_v5(model, roi_name, ROI_NNUNET_DIR,
                                         img_size=IMG_SIZE, n_slices=SLICE_CONTEXT, device=device)
        pt = (pred_vol == 2); gt = (gt_vol == 2)
        tp_t += (pt & gt).sum(); fp_t += (pt & ~gt).sum(); fn_t += (~pt & gt).sum()
        pm = (pred_vol >= 1); gm = (gt_vol >= 1)
        tp_m += (pm & gm).sum(); fp_m += (pm & ~gm).sum(); fn_m += (~pm & gm).sum()
        del pred_vol, gt_vol
    
    torch.cuda.empty_cache()
    kid = float(2*tp_m / max(2*tp_m+fp_m+fn_m, 1))
    tum = float(2*tp_t / max(2*tp_t+fp_t+fn_t, 1))
    results.append({'case_name': case_name, 'kidney_merged': kid, 'tumor_dice': tum})
    if (ci+1) % 10 == 0 or ci == 0:
        print(f"  [{ci+1}/{len(cases_tumor)}] {case_name}: KidM={kid:.4f} TumD={tum:.4f} ({time.time()-t0:.0f}s)")

ek = np.mean([r['kidney_merged'] for r in results])
et = np.mean([r['tumor_dice'] for r in results])

print(f"\n{'='*70}")
print(f"NB07e E2E RESULTS (100% Predicted ROI Training)")
print(f"{'='*70}")
print(f"                                Kidney(KiTS)  Tumor(case)")
print(f"  NB07e ep60 (pred ROI val):    {ek:.4f}        {et:.4f}")
print(f"  NB04n ep66 (GT+mixed):        0.9507        0.6817")
print(f"  Old ensemble E2E:             0.9348        0.7039")
print(f"  NB04d E2E (old Stage1):       0.9196        0.6876")

# Hopeless cases
print(f"\nHopeless cases:")
for r in results:
    if r['case_name'] in ['case_00088','case_00180','case_00102','case_00025','case_00070','case_00187']:
        print(f"  {r['case_name']}: KidM={r['kidney_merged']:.4f} TumD={r['tumor_dice']:.4f}")

# Previously collapsed
print(f"\nPreviously collapsed:")
for r in results:
    if r['case_name'] in ['case_00005','case_00150','case_00207','case_00020','case_00019']:
        print(f"  {r['case_name']}: KidM={r['kidney_merged']:.4f} TumD={r['tumor_dice']:.4f}")

with open(OUTPUT_DIR / 'results' / 'fold0_nb07e_e2e.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f"\nSaved.")

  TumorAwareHead bottleneck: 448 channels
  TumorAwareHead bottleneck: 448 channels
Loaded: epoch 60, TumD=0.8820
Val cases with tumor: 42
  [1/42] case_00000: KidM=0.9823 TumD=0.7861 (6s)
  [10/42] case_00055: KidM=0.9572 TumD=0.9321 (53s)
  [20/42] case_00096: KidM=0.9768 TumD=0.5706 (101s)
  [30/42] case_00131: KidM=0.9829 TumD=0.9337 (150s)
  [40/42] case_00196: KidM=0.9844 TumD=0.9580 (195s)

NB07e E2E RESULTS (100% Predicted ROI Training)
                                Kidney(KiTS)  Tumor(case)
  NB07e ep60 (pred ROI val):    0.9614        0.7710
  NB04n ep66 (GT+mixed):        0.9507        0.6817
  Old ensemble E2E:             0.9348        0.7039
  NB04d E2E (old Stage1):       0.9196        0.6876

Hopeless cases:
  case_00025: KidM=0.9791 TumD=0.9636
  case_00180: KidM=0.9437 TumD=0.7949

Previously collapsed:
  case_00020: KidM=0.9682 TumD=0.3456

Saved.


In [19]:
# ======== NB07e: TTA + PostProc + Failure Analysis (42 val cases) ========
import numpy as np, json, time, torch
import torch.nn.functional as F
from pathlib import Path
from scipy.ndimage import gaussian_filter, label as scipy_label

OUTPUT_DIR = Path(r"D:\fsdf_uca_kits\outputs")
ROI_NNUNET = OUTPUT_DIR / "kidney_rois_nnunet_80_20"
FOLD_PATH = OUTPUT_DIR / "stats" / "fold_splits_80_20.json"
PREPROC_DIR = OUTPUT_DIR / "preprocessed"

with open(FOLD_PATH) as f:
    splits = json.load(f)
val_cases = sorted(splits['folds']['fold_0']['val'])

with open(ROI_NNUNET / "roi_meta_all.json") as f:
    metas = json.load(f)
val_metas = [m for m in metas if m['case_name'] in set(val_cases)]
cases_tumor = sorted(set(m['case_name'] for m in val_metas if m['has_tumor']))
print(f"Val cases with tumor: {len(cases_tumor)}")

def infer_slice(model, sq, img_size, device):
    inp = F.interpolate(torch.from_numpy(sq[None]).float().to(device),
                        (img_size, img_size), mode='bilinear', align_corners=False)
    logits = model(inp)['logits']
    return torch.softmax(logits, dim=1).cpu().numpy()[0]

def infer_roi_tta(model, roi_name, roi_dir, img_size=256, n_slices=7, device='cuda'):
    img = np.load(roi_dir / f'{roi_name}_img.npy')
    seg = np.load(roi_dir / f'{roi_name}_seg.npy')
    D, H, W = img.shape
    half = n_slices // 2
    prob_vol = np.zeros((3, D, H, W), dtype=np.float32)
    max_dim = max(H, W)
    pad_h, pad_w = (max_dim - H) // 2, (max_dim - W) // 2
    pad_val = float(img.min())

    model.eval()
    with torch.no_grad():
        for s in range(D):
            slices = np.zeros((n_slices, H, W), dtype=np.float32)
            for j in range(n_slices):
                si = max(0, min(D-1, s - half + j))
                slices[j] = img[si]
            sq = np.full((n_slices, max_dim, max_dim), pad_val, dtype=np.float32)
            sq[:, pad_h:pad_h+H, pad_w:pad_w+W] = slices

            probs_sum = np.zeros((3, img_size, img_size), dtype=np.float32)
            for flip_lr, flip_ud in [(False,False),(True,False),(False,True),(True,True)]:
                sq_aug = sq.copy()
                if flip_lr: sq_aug = sq_aug[:, :, ::-1].copy()
                if flip_ud: sq_aug = sq_aug[:, ::-1, :].copy()
                p = infer_slice(model, sq_aug, img_size, device)
                if flip_lr: p = p[:, :, ::-1].copy()
                if flip_ud: p = p[:, ::-1, :].copy()
                probs_sum += p
            probs_avg = probs_sum / 4.0

            probs_full = np.zeros((3, max_dim, max_dim), dtype=np.float32)
            for c in range(3):
                probs_full[c] = F.interpolate(
                    torch.from_numpy(probs_avg[c:c+1, None]),
                    (max_dim, max_dim), mode='bilinear', align_corners=False
                ).numpy()[0, 0]
            prob_vol[:, s] = probs_full[:, pad_h:pad_h+H, pad_w:pad_w+W]

    for c in range(3):
        prob_vol[c] = gaussian_filter(prob_vol[c], sigma=[1.0, 0.5, 0.5])
    pred_vol = prob_vol.argmax(axis=0).astype(np.uint8)
    return pred_vol, seg

def postprocess(pred):
    merged = (pred >= 1).astype(np.int32)
    labeled, n = scipy_label(merged)
    if n > 2:
        sizes = [(labeled == i).sum() for i in range(1, n+1)]
        keep = np.argsort(sizes)[::-1][:2]
        mask = np.zeros_like(merged, dtype=bool)
        for i in keep: mask |= (labeled == (i+1))
        pred[~mask] = 0
    tumor = (pred == 2).astype(np.int32)
    t_lab, nt = scipy_label(tumor)
    for i in range(1, nt+1):
        if (t_lab == i).sum() < 50:
            pred[t_lab == i] = 1
    return pred

# Also run NO-TTA for direct comparison
def infer_roi_no_tta(model, roi_name, roi_dir, img_size=256, n_slices=7, device='cuda'):
    img = np.load(roi_dir / f'{roi_name}_img.npy')
    seg = np.load(roi_dir / f'{roi_name}_seg.npy')
    D, H, W = img.shape
    half = n_slices // 2
    prob_vol = np.zeros((3, D, H, W), dtype=np.float32)
    max_dim = max(H, W)
    pad_h, pad_w = (max_dim - H) // 2, (max_dim - W) // 2
    pad_val = float(img.min())
    model.eval()
    with torch.no_grad():
        for s in range(D):
            slices = np.zeros((n_slices, H, W), dtype=np.float32)
            for j in range(n_slices):
                si = max(0, min(D-1, s - half + j))
                slices[j] = img[si]
            sq = np.full((n_slices, max_dim, max_dim), pad_val, dtype=np.float32)
            sq[:, pad_h:pad_h+H, pad_w:pad_w+W] = slices
            p = infer_slice(model, sq, img_size, device)
            probs_full = np.zeros((3, max_dim, max_dim), dtype=np.float32)
            for c in range(3):
                probs_full[c] = F.interpolate(
                    torch.from_numpy(p[c:c+1, None]),
                    (max_dim, max_dim), mode='bilinear', align_corners=False
                ).numpy()[0, 0]
            prob_vol[:, s] = probs_full[:, pad_h:pad_h+H, pad_w:pad_w+W]
    for c in range(3):
        prob_vol[c] = gaussian_filter(prob_vol[c], sigma=[1.0, 0.5, 0.5])
    pred_vol = prob_vol.argmax(axis=0).astype(np.uint8)
    return pred_vol, seg

# Run both no-TTA and TTA+PP
results = []
t0 = time.time()

for ci, case_name in enumerate(cases_tumor):
    pred_rois = sorted(ROI_NNUNET.glob(f"{case_name}_roi*_img.npy"))
    # No-TTA accumulators
    tp_mn, fp_mn, fn_mn, tp_tn, fp_tn, fn_tn = 0,0,0,0,0,0
    # TTA+PP accumulators
    tp_m, fp_m, fn_m, tp_t, fp_t, fn_t = 0,0,0,0,0,0

    for p in pred_rois:
        roi_name = p.stem.replace('_img', '')
        if not (ROI_NNUNET / f'{roi_name}_seg.npy').exists(): continue

        # No TTA
        pred_n, gt_n = infer_roi_no_tta(model, roi_name, ROI_NNUNET,
                                         img_size=IMG_SIZE, n_slices=SLICE_CONTEXT, device=device)
        pred_n = postprocess(pred_n.copy())
        ptn = (pred_n == 2); gtn = (gt_n == 2)
        tp_tn += int((ptn & gtn).sum()); fp_tn += int((ptn & ~gtn).sum()); fn_tn += int((~ptn & gtn).sum())
        pmn = (pred_n >= 1); gmn = (gt_n >= 1)
        tp_mn += int((pmn & gmn).sum()); fp_mn += int((pmn & ~gmn).sum()); fn_mn += int((~pmn & gmn).sum())

        # TTA + PP
        pred_vol, gt_vol = infer_roi_tta(model, roi_name, ROI_NNUNET,
                                          img_size=IMG_SIZE, n_slices=SLICE_CONTEXT, device=device)
        pred_vol = postprocess(pred_vol.copy())
        pt = (pred_vol == 2); gt = (gt_vol == 2)
        tp_t += int((pt & gt).sum()); fp_t += int((pt & ~gt).sum()); fn_t += int((~pt & gt).sum())
        pm = (pred_vol >= 1); gm = (gt_vol >= 1)
        tp_m += int((pm & gm).sum()); fp_m += int((pm & ~gm).sum()); fn_m += int((~pm & gm).sum())

        del pred_vol, pred_n, gt_vol, gt_n

    torch.cuda.empty_cache()

    kid_n = float(2*tp_mn / max(2*tp_mn+fp_mn+fn_mn, 1))
    tum_n = float(2*tp_tn / max(2*tp_tn+fp_tn+fn_tn, 1))
    kid_t = float(2*tp_m / max(2*tp_m+fp_m+fn_m, 1))
    tum_t = float(2*tp_t / max(2*tp_t+fp_t+fn_t, 1))
    prec = float(tp_t / max(tp_t+fp_t, 1))
    rec = float(tp_t / max(tp_t+fn_t, 1))

    label = np.load(PREPROC_DIR / case_name / "label.npy")
    tvox = int((label == 2).sum())

    results.append({'case': case_name, 'kid_noTTA': kid_n, 'tum_noTTA': tum_n,
                    'kid_tta': kid_t, 'tum_tta': tum_t, 'prec': prec, 'rec': rec, 'tvox': tvox})
    if (ci+1) % 10 == 0 or ci == 0:
        print(f"  [{ci+1}/{len(cases_tumor)}] {case_name}: noTTA={tum_n:.4f} TTA+PP={tum_t:.4f} ({time.time()-t0:.0f}s)")

# Summary
ek_n = np.mean([r['kid_noTTA'] for r in results])
et_n = np.mean([r['tum_noTTA'] for r in results])
ek_t = np.mean([r['kid_tta'] for r in results])
et_t = np.mean([r['tum_tta'] for r in results])

print(f"\n{'='*70}")
print(f"NB07e FINAL RESULTS (80/20 split, ep60)")
print(f"{'='*70}")
print(f"  NB07e no TTA+PP:   Kidney {ek_n:.4f}  Tumor {et_n:.4f}")
print(f"  NB07e TTA+PP:      Kidney {ek_t:.4f}  Tumor {et_t:.4f}")
print(f"  TTA gain:          Kidney {ek_t-ek_n:+.4f}  Tumor {et_t-et_n:+.4f}")

print(f"\n  COMPARISON (best of each):")
print(f"  NB07d V5 (TTA+PP):    Kidney 0.9666  Tumor 0.7716")
best_e = max(et_n, et_t)
best_label = "no TTA" if et_n >= et_t else "TTA+PP"
print(f"  NB07e V6 ({best_label}):  Kidney {max(ek_n,ek_t):.4f}  Tumor {best_e:.4f}")
print(f"  V6 vs V5:             Tumor {best_e - 0.7716:+.4f}")

# Failure analysis
sorted_r = sorted(results, key=lambda r: max(r['tum_noTTA'], r['tum_tta']))

print(f"\n{'='*70}")
print(f"WORST 10 CASES (by best of noTTA/TTA)")
print(f"{'='*70}")
print(f"{'Case':<15} {'noTTA':>8} {'TTA+PP':>8} {'Prec':>6} {'Rec':>6} {'TumVox':>8} {'Issue':>20}")
print("-" * 75)
for r in sorted_r[:10]:
    best = max(r['tum_noTTA'], r['tum_tta'])
    if best == 0: issue = "ZERO (total miss)"
    elif r['rec'] < 0.3: issue = "LOW RECALL"
    elif r['prec'] < 0.3: issue = "FALSE POSITIVES"
    elif r['tvox'] < 3000: issue = f"TINY tumor"
    elif best < 0.5: issue = "BOUNDARY error"
    else: issue = "moderate"
    print(f"  {r['case']:<15} {r['tum_noTTA']:>7.4f} {r['tum_tta']:>7.4f} {r['prec']:>5.3f} {r['rec']:>5.3f} {r['tvox']:>7,}  {issue}")

print(f"\nBEST 5 CASES")
for r in sorted_r[-5:]:
    print(f"  {r['case']:<15} {r['tum_noTTA']:>7.4f} {r['tum_tta']:>7.4f} {r['tvox']:>7,}")

# NB07d vs NB07e per-case comparison
print(f"\n{'='*70}")
print(f"NB07e vs NB07d: per-case changes")
print(f"{'='*70}")
nb07d_path = OUTPUT_DIR / 'results_80_20' / 'nb07d_tta_pp.json'
if nb07d_path.exists():
    with open(nb07d_path) as f:
        nb07d_results = json.load(f)
    d_map = {r['case']: r['tum'] for r in nb07d_results}
    improved, degraded = [], []
    for r in results:
        d_val = d_map.get(r['case'], None)
        if d_val is not None:
            best_e = max(r['tum_noTTA'], r['tum_tta'])
            diff = best_e - d_val
            if diff > 0.02: improved.append((r['case'], d_val, best_e, diff))
            elif diff < -0.02: degraded.append((r['case'], d_val, best_e, diff))
    
    print(f"  Improved (>+0.02): {len(improved)} cases")
    for c, d, e, diff in sorted(improved, key=lambda x: -x[3])[:5]:
        print(f"    {c}: NB07d={d:.4f} → NB07e={e:.4f} ({diff:+.4f})")
    print(f"  Degraded (<-0.02): {len(degraded)} cases")
    for c, d, e, diff in sorted(degraded, key=lambda x: x[3])[:5]:
        print(f"    {c}: NB07d={d:.4f} → NB07e={e:.4f} ({diff:+.4f})")
else:
    print("  nb07d_tta_pp.json not found — skip comparison")

# Difficulty breakdown
zero = [r for r in results if max(r['tum_noTTA'], r['tum_tta']) == 0]
hard = [r for r in results if 0 < max(r['tum_noTTA'], r['tum_tta']) < 0.4]
med = [r for r in results if 0.4 <= max(r['tum_noTTA'], r['tum_tta']) < 0.8]
easy = [r for r in results if max(r['tum_noTTA'], r['tum_tta']) >= 0.8]
print(f"\nDifficulty breakdown:")
print(f"  Zero (=0.0):      {len(zero)} cases")
print(f"  Hard (<0.4):      {len(hard)} cases")
print(f"  Medium (0.4-0.8): {len(med)} cases")
print(f"  Easy (>0.8):      {len(easy)} cases")

# Save
(OUTPUT_DIR / 'results_80_20').mkdir(exist_ok=True)
with open(OUTPUT_DIR / 'results_80_20' / 'nb07e_tta_pp.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f"\nSaved to results_80_20/nb07e_tta_pp.json")

Val cases with tumor: 42
  [1/42] case_00000: noTTA=0.7874 TTA+PP=0.8315 (23s)
  [10/42] case_00055: noTTA=0.9321 TTA+PP=0.9186 (253s)
  [20/42] case_00096: noTTA=0.5706 TTA+PP=0.5955 (491s)
  [30/42] case_00131: noTTA=0.9337 TTA+PP=0.9384 (719s)
  [40/42] case_00196: noTTA=0.9580 TTA+PP=0.9481 (927s)

NB07e FINAL RESULTS (80/20 split, ep60)
  NB07e no TTA+PP:   Kidney 0.9614  Tumor 0.7713
  NB07e TTA+PP:      Kidney 0.9623  Tumor 0.7672
  TTA gain:          Kidney +0.0009  Tumor -0.0041

  COMPARISON (best of each):
  NB07d V5 (TTA+PP):    Kidney 0.9666  Tumor 0.7716
  NB07e V6 (no TTA):  Kidney 0.9623  Tumor 0.7713
  V6 vs V5:             Tumor -0.0003

WORST 10 CASES (by best of noTTA/TTA)
Case               noTTA   TTA+PP   Prec    Rec   TumVox                Issue
---------------------------------------------------------------------------
  case_00208       0.0000  0.0000 0.000 0.000   1,916  ZERO (total miss)
  case_00045       0.2569  0.2070 0.319 0.153  35,632  LOW RECALL
  cas

In [24]:
# ======== V5(TTA) + V6(noTTA) Ensemble — SELF-CONTAINED ========
import numpy as np, json, time, torch, torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp
from pathlib import Path
from scipy.ndimage import gaussian_filter, label as scipy_label

OUTPUT_DIR = Path(r"D:\fsdf_uca_kits\outputs")
ROI_NNUNET = OUTPUT_DIR / "kidney_rois_nnunet_80_20"
FOLD_PATH = OUTPUT_DIR / "stats" / "fold_splits_80_20.json"
PREPROC_DIR = OUTPUT_DIR / "preprocessed"
CKPT_DIR = OUTPUT_DIR / "checkpoints_80_20"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SLICE_CONTEXT = 7
NUM_CLASSES = 3
FUSE_CHANNELS = 48
ENCODER = 'efficientnet-b4'
IMG_SIZE = 256

# ---- V5 Architecture (for NB07d checkpoint) ----
class ASPP_ens(nn.Module):
    def __init__(self, in_channels, out_channels, rates=[1, 6, 12, 18]):
        super().__init__()
        self.convs = nn.ModuleList()
        for rate in rates:
            self.convs.append(nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=rate, dilation=rate),
                nn.InstanceNorm2d(out_channels), nn.ReLU(inplace=True)))
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_channels, out_channels, 1), nn.ReLU(True))
        self.fuse = nn.Sequential(
            nn.Conv2d(out_channels * (len(rates) + 1), out_channels, 1),
            nn.InstanceNorm2d(out_channels), nn.ReLU(True))
    def forward(self, x):
        feats = [conv(x) for conv in self.convs]
        feats.append(F.interpolate(self.gap(x), size=x.shape[2:], mode='bilinear', align_corners=False))
        return self.fuse(torch.cat(feats, dim=1))

class CBAM_ens(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        r = channels // reduction
        self.ca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(channels, r), nn.ReLU(True), nn.Linear(r, channels))
        self.ca_max = nn.Sequential(nn.AdaptiveMaxPool2d(1), nn.Flatten(), nn.Linear(channels, r), nn.ReLU(True), nn.Linear(r, channels))
        self.sa = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3), nn.Sigmoid())
        self.spatial_attn = None
    def forward(self, x):
        x = x * torch.sigmoid(self.ca(x) + self.ca_max(x)).unsqueeze(-1).unsqueeze(-1)
        sa = self.sa(torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], 1))
        self.spatial_attn = sa.detach()
        return x * sa

class FSDFUCANetV5_ens(nn.Module):
    def __init__(self, in_channels=7, num_classes=3, fuse_channels=48, encoder_name='efficientnet-b4'):
        super().__init__()
        self.spatial_unet = smp.UnetPlusPlus(encoder_name=encoder_name, encoder_weights='imagenet',
            in_channels=in_channels, classes=fuse_channels)
        self.aspp = ASPP_ens(fuse_channels, fuse_channels, rates=[1, 6, 12, 18])
        self.cbam = CBAM_ens(channels=fuse_channels, reduction=8)
        self.seg_head = nn.Sequential(nn.Conv2d(fuse_channels, fuse_channels, 3, padding=1),
            nn.InstanceNorm2d(fuse_channels), nn.ReLU(True), nn.Conv2d(fuse_channels, num_classes, 1))
        self.ds_head = nn.Conv2d(fuse_channels, num_classes, 1)
        self.edl_head = nn.Sequential(nn.Conv2d(fuse_channels, fuse_channels, 3, padding=1),
            nn.InstanceNorm2d(fuse_channels), nn.ReLU(True), nn.Conv2d(fuse_channels, num_classes, 1), nn.Softplus())
    def forward(self, x, return_uncertainty=False, return_ds=False):
        spatial_feat = self.spatial_unet(x)
        fused = self.aspp(spatial_feat)
        refined = self.cbam(fused)
        logits = self.seg_head(refined)
        outputs = [logits]
        if return_ds: outputs.append(self.ds_head(fused))
        if return_uncertainty:
            evidence = self.edl_head(refined)
            alpha = evidence + 1.0
            uncertainty = num_classes / alpha.sum(dim=1, keepdim=True)
            outputs.extend([evidence, uncertainty])
        return outputs[0] if len(outputs) == 1 else tuple(outputs)

# ---- V6 Architecture (for NB07e checkpoint) ----
class SliceAttentionGate_ens(nn.Module):
    def __init__(self, num_slices=7, pos_kernel=7):
        super().__init__()
        self.num_slices = num_slices
        pad = pos_kernel // 2
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=pos_kernel, padding=pad, bias=True)
        self.slice_conv = nn.Conv2d(num_slices, num_slices, kernel_size=1, bias=True)
        nn.init.zeros_(self.spatial_conv.weight); nn.init.zeros_(self.spatial_conv.bias)
        nn.init.zeros_(self.slice_conv.weight); nn.init.zeros_(self.slice_conv.bias)
    def forward(self, x):
        max_pool = x.max(dim=1, keepdim=True)[0]
        avg_pool = x.mean(dim=1, keepdim=True)
        spatial_logits = self.spatial_conv(torch.cat([max_pool, avg_pool], dim=1))
        spatial_gate = 0.5 + torch.sigmoid(spatial_logits)
        x = x * spatial_gate
        slice_logits = self.slice_conv(x)
        slice_gate = torch.softmax(slice_logits, dim=1) * self.num_slices
        x = x * slice_gate
        return x

class TumorAwareHead_ens(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.head = nn.Sequential(nn.Conv2d(in_channels, 64, 1), nn.ReLU(inplace=True), nn.Conv2d(64, 1, 1))
    def forward(self, bottleneck):
        return self.head(bottleneck)

class FSDFUCANetV6_ens(nn.Module):
    def __init__(self, in_channels=7, num_classes=3, fuse_channels=48, encoder_name='efficientnet-b4'):
        super().__init__()
        self.slice_gate = SliceAttentionGate_ens(num_slices=in_channels)
        self.spatial_unet = smp.UnetPlusPlus(encoder_name=encoder_name, encoder_weights='imagenet',
            in_channels=in_channels, classes=fuse_channels)
        self.aspp = ASPP_ens(fuse_channels, fuse_channels, rates=[1, 6, 12, 18])
        self.cbam = CBAM_ens(channels=fuse_channels, reduction=8)
        self.seg_head = nn.Sequential(nn.Conv2d(fuse_channels, fuse_channels, 3, padding=1),
            nn.InstanceNorm2d(fuse_channels), nn.ReLU(True), nn.Conv2d(fuse_channels, num_classes, 1))
        self.ds_head = nn.Conv2d(fuse_channels, num_classes, 1)
        self.edl_head = nn.Sequential(nn.Conv2d(fuse_channels, fuse_channels, 3, padding=1),
            nn.InstanceNorm2d(fuse_channels), nn.ReLU(True), nn.Conv2d(fuse_channels, num_classes, 1), nn.Softplus())
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, 64, 64)
            feats = self.spatial_unet.encoder(dummy)
            bottleneck_ch = feats[-1].shape[1]
        self.tumor_head = TumorAwareHead_ens(in_channels=bottleneck_ch)
    def forward(self, x, return_uncertainty=False, return_ds=False):
        x = self.slice_gate(x)
        encoder_features = self.spatial_unet.encoder(x)
        tumor_logits = self.tumor_head(encoder_features[-1])
        decoder_out = self.spatial_unet.decoder(encoder_features)
        spatial_feat = self.spatial_unet.segmentation_head(decoder_out)
        fused = self.aspp(spatial_feat)
        refined = self.cbam(fused)
        logits = self.seg_head(refined)
        out = {'logits': logits, 'tumor_logits': tumor_logits}
        if return_ds: out['ds_logits'] = self.ds_head(fused)
        if return_uncertainty:
            evidence = self.edl_head(refined)
            alpha = evidence + 1.0
            uncertainty = num_classes / alpha.sum(dim=1, keepdim=True)
            out['evidence'] = evidence; out['uncertainty'] = uncertainty
        return out

# ---- Load both models ----
print("Loading V5 (NB07d)...")
ens_v5 = FSDFUCANetV5_ens(SLICE_CONTEXT, NUM_CLASSES, FUSE_CHANNELS, ENCODER).to(device)
ckpt_v5 = torch.load(CKPT_DIR / 'nb07d_fold0_best.pth', map_location=device, weights_only=False)
ens_v5.load_state_dict(ckpt_v5['model_state'])
ens_v5.eval()
print(f"  V5: ep{ckpt_v5['epoch']}, TumD={ckpt_v5['tumor_dice']:.4f}")

print("Loading V6 (NB07e)...")
ens_v6 = FSDFUCANetV6_ens(SLICE_CONTEXT, NUM_CLASSES, FUSE_CHANNELS, ENCODER).to(device)
ckpt_v6 = torch.load(CKPT_DIR / 'nb07e_fold0_best.pth', map_location=device, weights_only=False)
ens_v6.load_state_dict(ckpt_v6['model_state'])
ens_v6.eval()
print(f"  V6: ep{ckpt_v6['epoch']}, TumD={ckpt_v6['tumor_dice']:.4f}")

# ---- Data ----
with open(FOLD_PATH) as f:
    splits = json.load(f)
val_cases = sorted(splits['folds']['fold_0']['val'])
with open(ROI_NNUNET / "roi_meta_all.json") as f:
    metas = json.load(f)
val_metas = [m for m in metas if m['case_name'] in set(val_cases)]
cases_tumor = sorted(set(m['case_name'] for m in val_metas if m['has_tumor']))
print(f"Val cases with tumor: {len(cases_tumor)}")

# ---- Inference ----
def infer_slice_simple(model, sq, img_size, device, is_v6=False):
    inp = F.interpolate(torch.from_numpy(sq[None]).float().to(device),
                        (img_size, img_size), mode='bilinear', align_corners=False)
    out = model(inp)
    if is_v6:
        logits = out['logits']
    else:
        logits = out if not isinstance(out, tuple) else out[0]
    return torch.softmax(logits, dim=1).cpu().numpy()[0]

def infer_roi_ensemble(mv5, mv6, roi_name, roi_dir, img_size=256, n_slices=7, device='cuda'):
    img = np.load(roi_dir / f'{roi_name}_img.npy')
    seg = np.load(roi_dir / f'{roi_name}_seg.npy')
    D, H, W = img.shape
    half = n_slices // 2
    prob_vol = np.zeros((3, D, H, W), dtype=np.float32)
    max_dim = max(H, W)
    pad_h, pad_w = (max_dim - H) // 2, (max_dim - W) // 2
    pad_val = float(img.min())

    with torch.no_grad():
        for s in range(D):
            slices = np.zeros((n_slices, H, W), dtype=np.float32)
            for j in range(n_slices):
                si = max(0, min(D-1, s - half + j))
                slices[j] = img[si]
            sq = np.full((n_slices, max_dim, max_dim), pad_val, dtype=np.float32)
            sq[:, pad_h:pad_h+H, pad_w:pad_w+W] = slices

            probs_sum = np.zeros((3, img_size, img_size), dtype=np.float32)

            # V5: 4-flip TTA (TTA helps V5)
            for flip_lr, flip_ud in [(False,False),(True,False),(False,True),(True,True)]:
                sq_aug = sq.copy()
                if flip_lr: sq_aug = sq_aug[:, :, ::-1].copy()
                if flip_ud: sq_aug = sq_aug[:, ::-1, :].copy()
                p5 = infer_slice_simple(mv5, sq_aug, img_size, device, is_v6=False)
                if flip_lr: p5 = p5[:, :, ::-1].copy()
                if flip_ud: p5 = p5[:, ::-1, :].copy()
                probs_sum += p5

            # V6: NO TTA (TTA hurts V6)
            p6 = infer_slice_simple(mv6, sq, img_size, device, is_v6=True)
            probs_sum += p6

            probs_avg = probs_sum / 5.0  # 4 V5 flips + 1 V6

            probs_full = np.zeros((3, max_dim, max_dim), dtype=np.float32)
            for c in range(3):
                probs_full[c] = F.interpolate(
                    torch.from_numpy(probs_avg[c:c+1, None]),
                    (max_dim, max_dim), mode='bilinear', align_corners=False
                ).numpy()[0, 0]
            prob_vol[:, s] = probs_full[:, pad_h:pad_h+H, pad_w:pad_w+W]

    for c in range(3):
        prob_vol[c] = gaussian_filter(prob_vol[c], sigma=[1.0, 0.5, 0.5])
    return prob_vol.argmax(axis=0).astype(np.uint8), seg

def postprocess(pred):
    merged = (pred >= 1).astype(np.int32)
    labeled, n = scipy_label(merged)
    if n > 2:
        sizes = [(labeled == i).sum() for i in range(1, n+1)]
        keep = np.argsort(sizes)[::-1][:2]
        mask = np.zeros_like(merged, dtype=bool)
        for i in keep: mask |= (labeled == (i+1))
        pred[~mask] = 0
    tumor = (pred == 2).astype(np.int32)
    t_lab, nt = scipy_label(tumor)
    for i in range(1, nt+1):
        if (t_lab == i).sum() < 50:
            pred[t_lab == i] = 1
    return pred

# ---- Run ensemble ----
results = []
t0 = time.time()

for ci, case_name in enumerate(cases_tumor):
    pred_rois = sorted(ROI_NNUNET.glob(f"{case_name}_roi*_img.npy"))
    tp_m, fp_m, fn_m, tp_t, fp_t, fn_t = 0,0,0,0,0,0

    for p in pred_rois:
        roi_name = p.stem.replace('_img', '')
        if not (ROI_NNUNET / f'{roi_name}_seg.npy').exists(): continue
        pred_vol, gt_vol = infer_roi_ensemble(
            ens_v5, ens_v6, roi_name, ROI_NNUNET,
            img_size=IMG_SIZE, n_slices=SLICE_CONTEXT, device=device)
        pred_vol = postprocess(pred_vol.copy())

        pt = (pred_vol == 2); gt = (gt_vol == 2)
        tp_t += int((pt & gt).sum()); fp_t += int((pt & ~gt).sum()); fn_t += int((~pt & gt).sum())
        pm = (pred_vol >= 1); gm = (gt_vol >= 1)
        tp_m += int((pm & gm).sum()); fp_m += int((pm & ~gm).sum()); fn_m += int((~pm & gm).sum())
        del pred_vol, gt_vol

    torch.cuda.empty_cache()
    kid = float(2*tp_m / max(2*tp_m+fp_m+fn_m, 1))
    tum = float(2*tp_t / max(2*tp_t+fp_t+fn_t, 1))
    prec = float(tp_t / max(tp_t+fp_t, 1))
    rec = float(tp_t / max(tp_t+fn_t, 1))
    label = np.load(PREPROC_DIR / case_name / "label.npy")
    tvox = int((label == 2).sum())

    results.append({'case': case_name, 'kid': kid, 'tum': tum,
                    'prec': prec, 'rec': rec, 'tvox': tvox})
    if (ci+1) % 10 == 0 or ci == 0:
        print(f"  [{ci+1}/{len(cases_tumor)}] {case_name}: KidM={kid:.4f} TumD={tum:.4f} ({time.time()-t0:.0f}s)")

ek = np.mean([r['kid'] for r in results])
et = np.mean([r['tum'] for r in results])

print(f"\n{'='*70}")
print(f"V5(TTA) + V6(noTTA) ENSEMBLE RESULTS (42 val cases)")
print(f"{'='*70}")
print(f"  Ensemble:               Kidney {ek:.4f}  Tumor {et:.4f}")
print(f"  NB07d V5 (TTA+PP):      Kidney 0.9666  Tumor 0.7716")
print(f"  NB07e V6 (noTTA+PP):    Kidney 0.9614  Tumor 0.7713")
print(f"  Ensemble vs V5:         Tumor {et - 0.7716:+.4f}")
print(f"  Ensemble vs V6:         Tumor {et - 0.7713:+.4f}")

# Per-case comparison
nb07d_path = OUTPUT_DIR / 'results_80_20' / 'nb07d_tta_pp.json'
if nb07d_path.exists():
    with open(nb07d_path) as f:
        d_data = json.load(f)
    d_map = {r['case']: r['tum'] for r in d_data}
    improved = [(r['case'], d_map.get(r['case'],0), r['tum']) for r in results
                if r['tum'] > d_map.get(r['case'],0) + 0.02]
    degraded = [(r['case'], d_map.get(r['case'],0), r['tum']) for r in results
                if r['tum'] < d_map.get(r['case'],0) - 0.02]
    print(f"\n  vs NB07d V5: Improved {len(improved)}, Degraded {len(degraded)}")
    for c, d, e in sorted(improved, key=lambda x: -(x[2]-x[1]))[:5]:
        print(f"    {c}: V5={d:.4f} Ens={e:.4f} ({e-d:+.4f})")
    if degraded:
        print(f"  Degraded:")
        for c, d, e in sorted(degraded, key=lambda x: x[2]-x[1])[:5]:
            print(f"    {c}: V5={d:.4f} Ens={e:.4f} ({e-d:+.4f})")

# Worst 5
sorted_r = sorted(results, key=lambda r: r['tum'])
print(f"\n  WORST 5:")
for r in sorted_r[:5]:
    d_val = d_map.get(r['case'], 0) if nb07d_path.exists() else 0
    print(f"    {r['case']}: Ens={r['tum']:.4f} V5={d_val:.4f} Prec={r['prec']:.3f} Rec={r['rec']:.3f}")

(OUTPUT_DIR / 'results_80_20').mkdir(exist_ok=True)
with open(OUTPUT_DIR / 'results_80_20' / 'nb07e_ensemble_v5v6.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f"\nSaved to results_80_20/nb07e_ensemble_v5v6.json")

Loading V5 (NB07d)...
  V5: ep63, TumD=0.8786
Loading V6 (NB07e)...
  V6: ep60, TumD=0.8820
Val cases with tumor: 42
  [1/42] case_00000: KidM=0.9838 TumD=0.8385 (20s)
  [10/42] case_00055: KidM=0.9616 TumD=0.9408 (231s)
  [20/42] case_00096: KidM=0.9776 TumD=0.6543 (436s)
  [30/42] case_00131: KidM=0.9835 TumD=0.9392 (648s)
  [40/42] case_00196: KidM=0.9873 TumD=0.9549 (848s)

V5(TTA) + V6(noTTA) ENSEMBLE RESULTS (42 val cases)
  Ensemble:               Kidney 0.9662  Tumor 0.7766
  NB07d V5 (TTA+PP):      Kidney 0.9666  Tumor 0.7716
  NB07e V6 (noTTA+PP):    Kidney 0.9614  Tumor 0.7713
  Ensemble vs V5:         Tumor +0.0050
  Ensemble vs V6:         Tumor +0.0053

  vs NB07d V5: Improved 5, Degraded 2
    case_00128: V5=0.6069 Ens=0.6541 (+0.0473)
    case_00069: V5=0.4732 Ens=0.5113 (+0.0381)
    case_00094: V5=0.6843 Ens=0.7219 (+0.0376)
    case_00176: V5=0.6544 Ens=0.6889 (+0.0344)
    case_00125: V5=0.7214 Ens=0.7493 (+0.0279)
  Degraded:
    case_00116: V5=0.5847 Ens=0.4964 (-

In [13]:
# ======== TTA + Post-Processing on NB07e ep60 ========
import numpy as np
import json, time
from pathlib import Path
from scipy.ndimage import gaussian_filter, label as scipy_label

FOLD_PATH = Path(r"D:\fsdf_uca_kits\outputs\stats\fold_splits_80_20.json")
ROI_NNUNET_DIR = OUTPUT_DIR / "kidney_rois_nnunet_80_20"

# Load best checkpoint
model = FSDFUCANetV6(SLICE_CONTEXT, NUM_CLASSES, FUSE_CHANNELS, ENCODER).to(device)
ckpt = torch.load(CKPT_DIR / 'nb07e_fold0_best.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Loaded: epoch {ckpt['epoch']}, TumD={ckpt['tumor_dice']:.4f}")

with open(FOLD_PATH) as f:
    splits = json.load(f)
val_case_names = set(splits['folds']['fold_0']['val'])

with open(ROI_NNUNET_DIR / "roi_meta_all.json") as f:
    nn_metas = json.load(f)
nn_val = [m for m in nn_metas if m['case_name'] in val_case_names]
cases_tumor = sorted(set(m['case_name'] for m in nn_val if m['has_tumor']))
print(f"Val cases: {len(cases_tumor)}")

def infer_slice_v5(model, sq_input, img_size, device):
    """Run inference on one squared slice input."""
    inp = torch.nn.functional.interpolate(
        torch.from_numpy(sq_input[None]).float().to(device),
        (img_size, img_size), mode='bilinear', align_corners=False)
    logits = model(inp)["logits"]
    if isinstance(logits, tuple):
        logits = logits[0]
    return torch.softmax(logits, dim=1).cpu().numpy()[0]

def infer_roi_tta_v5(model, roi_name, roi_dir, img_size=256, n_slices=7, device='cuda'):
    """Inference with TTA: original + flip LR + flip UD + flip both."""
    img = np.load(roi_dir / f'{roi_name}_img.npy')
    seg = np.load(roi_dir / f'{roi_name}_seg.npy')
    D, H, W = img.shape
    half = n_slices // 2
    prob_vol = np.zeros((3, D, H, W), dtype=np.float32)
    
    max_dim = max(H, W)
    pad_h = (max_dim - H) // 2
    pad_w = (max_dim - W) // 2
    pad_val = float(img.min())
    
    model.eval()
    with torch.no_grad():
        for s in range(D):
            slices = np.zeros((n_slices, H, W), dtype=np.float32)
            for j in range(n_slices):
                si = max(0, min(D-1, s - half + j))
                slices[j] = img[si]
            
            # Pad to square
            sq = np.full((n_slices, max_dim, max_dim), pad_val, dtype=np.float32)
            sq[:, pad_h:pad_h+H, pad_w:pad_w+W] = slices
            
            # TTA: 4 augmentations
            probs_sum = np.zeros((3, img_size, img_size), dtype=np.float32)
            
            # 1. Original
            probs_sum += infer_slice_v5(model, sq, img_size, device)
            
            # 2. Flip left-right
            sq_lr = sq[:, :, ::-1].copy()
            p_lr = infer_slice_v5(model, sq_lr, img_size, device)
            probs_sum += p_lr[:, :, ::-1]
            
            # 3. Flip up-down
            sq_ud = sq[:, ::-1, :].copy()
            p_ud = infer_slice_v5(model, sq_ud, img_size, device)
            probs_sum += p_ud[:, ::-1, :]
            
            # 4. Flip both
            sq_both = sq[:, ::-1, ::-1].copy()
            p_both = infer_slice_v5(model, sq_both, img_size, device)
            probs_sum += p_both[:, ::-1, ::-1]
            
            # Average
            probs_avg = probs_sum / 4.0
            
            # Resize back to square
            probs_full = np.zeros((3, max_dim, max_dim), dtype=np.float32)
            for c in range(3):
                probs_full[c] = torch.nn.functional.interpolate(
                    torch.from_numpy(probs_avg[c:c+1, None]),
                    (max_dim, max_dim), mode='bilinear', align_corners=False
                ).numpy()[0, 0]
            
            prob_vol[:, s] = probs_full[:, pad_h:pad_h+H, pad_w:pad_w+W]
    
    # 3D Gaussian smooth
    for c in range(3):
        prob_vol[c] = gaussian_filter(prob_vol[c], sigma=[1.0, 0.5, 0.5])
    
    pred_vol = prob_vol.argmax(axis=0).astype(np.uint8)
    return pred_vol, seg

def postprocess_prediction(pred_vol):
    """Connected component: keep largest kidney components, remove small tumor noise."""
    # Keep largest 2 kidney+tumor components
    merged = (pred_vol >= 1).astype(np.int32)
    labeled, n_comp = scipy_label(merged)
    if n_comp > 2:
        comp_sizes = [(labeled == i).sum() for i in range(1, n_comp + 1)]
        sorted_comps = np.argsort(comp_sizes)[::-1]
        keep_mask = np.zeros_like(merged, dtype=bool)
        for i in range(min(2, n_comp)):
            keep_mask |= (labeled == (sorted_comps[i] + 1))
        pred_vol[~keep_mask] = 0
    
    # Remove tiny tumor components (< 50 voxels)
    tumor_mask = (pred_vol == 2).astype(np.int32)
    tumor_labeled, n_tumor = scipy_label(tumor_mask)
    for i in range(1, n_tumor + 1):
        if (tumor_labeled == i).sum() < 50:
            pred_vol[tumor_labeled == i] = 1  # convert to kidney
    
    return pred_vol

# Run evaluation: No TTA, TTA, TTA+PostProc
results_no_tta = []
results_tta = []
results_tta_pp = []
t0 = time.time()

for ci, case_name in enumerate(cases_tumor):
    pred_rois = sorted(ROI_NNUNET_DIR.glob(f"{case_name}_roi*_img.npy"))
    tp_m, fp_m, fn_m, tp_t, fp_t, fn_t = 0,0,0,0,0,0
    tp_m2, fp_m2, fn_m2, tp_t2, fp_t2, fn_t2 = 0,0,0,0,0,0
    
    for p in pred_rois:
        roi_name = p.stem.replace('_img', '')
        if not (ROI_NNUNET_DIR / f'{roi_name}_seg.npy').exists(): continue
        
        pred_vol, gt_vol = infer_roi_tta_v5(
            model, roi_name, ROI_NNUNET_DIR,
            img_size=IMG_SIZE, n_slices=SLICE_CONTEXT, device=device)
        
        # TTA result
        pt = (pred_vol == 2); gt = (gt_vol == 2)
        tp_t += (pt & gt).sum(); fp_t += (pt & ~gt).sum(); fn_t += (~pt & gt).sum()
        pm = (pred_vol >= 1); gm = (gt_vol >= 1)
        tp_m += (pm & gm).sum(); fp_m += (pm & ~gm).sum(); fn_m += (~pm & gm).sum()
        
        # TTA + post-processing
        pred_pp = postprocess_prediction(pred_vol.copy())
        pt2 = (pred_pp == 2); gt2 = (gt_vol == 2)
        tp_t2 += (pt2 & gt2).sum(); fp_t2 += (pt2 & ~gt2).sum(); fn_t2 += (~pt2 & gt2).sum()
        pm2 = (pred_pp >= 1); gm2 = (gt_vol >= 1)
        tp_m2 += (pm2 & gm2).sum(); fp_m2 += (pm2 & ~gm2).sum(); fn_m2 += (~pm2 & gm2).sum()
        
        del pred_vol, pred_pp, gt_vol
    
    torch.cuda.empty_cache()
    kid = float(2*tp_m / max(2*tp_m+fp_m+fn_m, 1))
    tum = float(2*tp_t / max(2*tp_t+fp_t+fn_t, 1))
    results_tta.append({'case_name': case_name, 'kidney_merged': kid, 'tumor_dice': tum})
    
    kid2 = float(2*tp_m2 / max(2*tp_m2+fp_m2+fn_m2, 1))
    tum2 = float(2*tp_t2 / max(2*tp_t2+fp_t2+fn_t2, 1))
    results_tta_pp.append({'case_name': case_name, 'kidney_merged': kid2, 'tumor_dice': tum2})
    
    if (ci+1) % 10 == 0 or ci == 0:
        print(f"  [{ci+1}/{len(cases_tumor)}] {case_name}: TTA={tum:.4f} TTA+PP={tum2:.4f} ({time.time()-t0:.0f}s)")

ek_tta = np.mean([r['kidney_merged'] for r in results_tta])
et_tta = np.mean([r['tumor_dice'] for r in results_tta])
ek_pp = np.mean([r['kidney_merged'] for r in results_tta_pp])
et_pp = np.mean([r['tumor_dice'] for r in results_tta_pp])

print(f"\n{'='*70}")
print(f"NB07e E2E: No TTA vs TTA vs TTA+PostProc")
print(f"{'='*70}")
print(f"                                Kidney(KiTS)  Tumor(case)")
print(f"  NB07e (no TTA):               0.9566        0.6960")
print(f"  NB07e + TTA:                  {ek_tta:.4f}        {et_tta:.4f}")
print(f"  NB07e + TTA + PostProc:       {ek_pp:.4f}        {et_pp:.4f}")
print(f"  TTA gain:                     {ek_tta-0.9566:+.4f}        {et_tta-0.6960:+.4f}")
print(f"  PostProc gain:                {ek_pp-ek_tta:+.4f}        {et_pp-et_tta:+.4f}")
print(f"\n  Old ensemble E2E:             0.9348        0.7039")
print(f"  Santini 2.5D (official):      0.9600        0.7400")

with open(OUTPUT_DIR / 'results' / 'fold0_nb07e_tta_pp.json', 'w') as f:
    json.dump({'tta': results_tta, 'tta_pp': results_tta_pp}, f, indent=2, default=str)
print(f"\nSaved.")

  TumorAwareHead bottleneck: 448 channels
Loaded: epoch 60, TumD=0.8820
Val cases: 42
  [1/42] case_00000: TTA=0.8315 TTA+PP=0.8315 (18s)
  [10/42] case_00055: TTA=0.9186 TTA+PP=0.9186 (199s)
  [20/42] case_00096: TTA=0.5955 TTA+PP=0.5955 (388s)
  [30/42] case_00131: TTA=0.9384 TTA+PP=0.9384 (580s)
  [40/42] case_00196: TTA=0.9481 TTA+PP=0.9481 (760s)

NB07e E2E: No TTA vs TTA vs TTA+PostProc
                                Kidney(KiTS)  Tumor(case)
  NB07e (no TTA):               0.9566        0.6960
  NB07e + TTA:                  0.9623        0.7672
  NB07e + TTA + PostProc:       0.9623        0.7672
  TTA gain:                     +0.0057        +0.0712
  PostProc gain:                +0.0000        +0.0000

  Old ensemble E2E:             0.9348        0.7039
  Santini 2.5D (official):      0.9600        0.7400

Saved.


In [14]:
# ======== GT-ROI Evaluation: NB07e ep60 ========
import numpy as np, json, time
from pathlib import Path
from scipy.ndimage import gaussian_filter

ROI_GT_DIR = OUTPUT_DIR / "kidney_rois_gt"
FOLD_PATH = Path(r"D:\fsdf_uca_kits\outputs\stats\fold_splits_80_20.json")

with open(FOLD_PATH) as f:
    splits = json.load(f)
with open(ROI_GT_DIR / "roi_meta_all.json") as f:
    gt_metas = json.load(f)

val_cases = set(splits['folds']['fold_0']['val'])
gt_val = [m for m in gt_metas if m['case_name'] in val_cases and m['has_tumor']]
gt_case_names = sorted(set(m['case_name'] for m in gt_val))
print(f"GT-ROI val cases with tumor: {len(gt_case_names)}")

# Model already loaded (ep60)
results_gt = []
t0 = time.time()

for ci, case_name in enumerate(gt_case_names):
    pred_rois = sorted(ROI_GT_DIR.glob(f"{case_name}_roi*_img.npy"))
    tp_m, fp_m, fn_m, tp_t, fp_t, fn_t = 0,0,0,0,0,0
    
    for p in pred_rois:
        roi_name = p.stem.replace('_img', '')
        if not (ROI_GT_DIR / f'{roi_name}_seg.npy').exists(): continue
        
        img = np.load(ROI_GT_DIR / f'{roi_name}_img.npy')
        seg = np.load(ROI_GT_DIR / f'{roi_name}_seg.npy')
        D, H, W = img.shape
        half = SLICE_CONTEXT // 2
        prob_vol = np.zeros((3, D, H, W), dtype=np.float32)
        
        with torch.no_grad():
            for s in range(D):
                slices = np.zeros((SLICE_CONTEXT, H, W), dtype=np.float32)
                for j in range(SLICE_CONTEXT):
                    si = max(0, min(D-1, s - half + j))
                    slices[j] = img[si]
                
                max_dim = max(H, W)
                pad_h = (max_dim - H) // 2
                pad_w = (max_dim - W) // 2
                pad_val = float(slices.min())
                sq = np.full((SLICE_CONTEXT, max_dim, max_dim), pad_val, dtype=np.float32)
                sq[:, pad_h:pad_h+H, pad_w:pad_w+W] = slices
                
                inp = torch.from_numpy(sq[None]).float().to(device)
                inp = torch.nn.functional.interpolate(inp, (IMG_SIZE, IMG_SIZE),
                                                        mode='bilinear', align_corners=False)
                logits = model(inp)["logits"]
                if isinstance(logits, tuple): logits = logits[0]
                probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
                
                probs_full = np.zeros((3, max_dim, max_dim), dtype=np.float32)
                for c in range(3):
                    probs_full[c] = torch.nn.functional.interpolate(
                        torch.from_numpy(probs[c:c+1, None]),
                        (max_dim, max_dim), mode='bilinear', align_corners=False
                    ).numpy()[0, 0]
                prob_vol[:, s] = probs_full[:, pad_h:pad_h+H, pad_w:pad_w+W]
        
        for c in range(3):
            prob_vol[c] = gaussian_filter(prob_vol[c], sigma=[1.0, 0.5, 0.5])
        pred_vol = prob_vol.argmax(axis=0).astype(np.uint8)
        
        pt = (pred_vol == 2); gt = (seg == 2)
        tp_t += (pt & gt).sum(); fp_t += (pt & ~gt).sum(); fn_t += (~pt & gt).sum()
        pm = (pred_vol >= 1); gm = (seg >= 1)
        tp_m += (pm & gm).sum(); fp_m += (pm & ~gm).sum(); fn_m += (~pm & gm).sum()
    
    torch.cuda.empty_cache()
    kid = float(2*tp_m / max(2*tp_m+fp_m+fn_m, 1))
    tum = float(2*tp_t / max(2*tp_t+fp_t+fn_t, 1))
    results_gt.append({'case_name': case_name, 'kidney_merged': kid, 'tumor_dice': tum})
    if (ci+1) % 10 == 0 or ci == 0:
        print(f"  [{ci+1}/{len(gt_case_names)}] {case_name}: KidM={kid:.4f} TumD={tum:.4f} ({time.time()-t0:.0f}s)")

ek = np.mean([r['kidney_merged'] for r in results_gt])
et = np.mean([r['tumor_dice'] for r in results_gt])

print(f"\n{'='*70}")
print(f"GT-ROI COMPARISON")
print(f"{'='*70}")
print(f"                                Kidney(KiTS)  Tumor(case)")
print(f"  NB04d (trained GT, tested GT):  0.9711        0.7920")
print(f"  NB07e (trained pred, tested GT): {ek:.4f}       {et:.4f}")
print(f"  NB07e E2E (tested pred):         0.9566        0.6960")
print(f"  Difference (NB07e GT vs NB04d):  {ek-0.9711:+.4f}       {et-0.7920:+.4f}")

GT-ROI val cases with tumor: 42
  [1/42] case_00000: KidM=0.9794 TumD=0.5487 (5s)
  [10/42] case_00055: KidM=0.9401 TumD=0.9189 (48s)
  [20/42] case_00096: KidM=0.9677 TumD=0.6473 (94s)
  [30/42] case_00131: KidM=0.9760 TumD=0.9294 (139s)
  [40/42] case_00196: KidM=0.9822 TumD=0.9484 (184s)

GT-ROI COMPARISON
                                Kidney(KiTS)  Tumor(case)
  NB04d (trained GT, tested GT):  0.9711        0.7920
  NB07e (trained pred, tested GT): 0.9527       0.7708
  NB07e E2E (tested pred):         0.9566        0.6960
  Difference (NB07e GT vs NB04d):  -0.0184       -0.0212


In [15]:
import numpy as np
import json
from pathlib import Path

with open(Path(r"D:\fsdf_uca_kits\outputs\results\fold0_nb07e_tta_pp.json")) as f:
    data = json.load(f)

results = data['tta_pp']

scores = sorted([(r['case_name'], r['tumor_dice']) for r in results], key=lambda x: x[1])

print("ALL 42 CASES RANKED BY TUMOR DICE:")
print(f"{'Case':<15} {'TumD':>8} {'Impact on mean':>15}")
print("-" * 40)
for name, score in scores[:15]:
    # How much this case drags the mean
    mean_without = np.mean([s for n, s in scores if n != name])
    mean_with = np.mean([s for _, s in scores])
    impact = mean_with - mean_without
    print(f"  {name:<15} {score:>8.4f} {impact:>+14.4f}")

print(f"\n...")
for name, score in scores[-5:]:
    print(f"  {name:<15} {score:>8.4f}")

# Simulate different val set sizes
all_scores = [s for _, s in scores]
print(f"\n{'='*50}")
print(f"SIMULATION: What if fewer hard cases in val?")
print(f"{'='*50}")
print(f"  All 42 cases:              {np.mean(all_scores):.4f}")
print(f"  Remove 1 worst:            {np.mean(all_scores[1:]):.4f}")
print(f"  Remove 3 worst:            {np.mean(all_scores[3:]):.4f}")
print(f"  Remove 5 worst:            {np.mean(all_scores[5:]):.4f}")

# Simulate 42-case val (take every case, pick best 42)
# Random sample 42 from 70, repeat 100 times
np.random.seed(42)
sim_means = []
for _ in range(1000):
    sample = np.random.choice(all_scores, size=42, replace=False)
    sim_means.append(np.mean(sample))

print(f"\n  Random 42-case sample (1000 trials):")
print(f"    Mean of means:    {np.mean(sim_means):.4f}")
print(f"    Best 42-case:     {np.max(sim_means):.4f}")
print(f"    Worst 42-case:    {np.min(sim_means):.4f}")
print(f"    Std:              {np.std(sim_means):.4f}")

# Stratified split: proportional hard/medium/easy
hard = [s for _, s in scores if s < 0.40]
medium = [s for _, s in scores if 0.40 <= s < 0.80]
easy = [s for _, s in scores if s >= 0.80]
print(f"\n  Difficulty distribution:")
print(f"    Hard (<0.40):   {len(hard)} cases, mean {np.mean(hard):.4f}" if hard else "    Hard: 0")
print(f"    Medium (0.4-0.8): {len(medium)} cases, mean {np.mean(medium):.4f}" if medium else "    Medium: 0")
print(f"    Easy (>0.80):   {len(easy)} cases, mean {np.mean(easy):.4f}" if easy else "    Easy: 0")

ALL 42 CASES RANKED BY TUMOR DICE:
Case                TumD  Impact on mean
----------------------------------------
  case_00208        0.0000        -0.0187
  case_00020        0.1781        -0.0144
  case_00045        0.2070        -0.0137
  case_00116        0.4095        -0.0087
  case_00133        0.5664        -0.0049
  case_00096        0.5955        -0.0042
  case_00075        0.6131        -0.0038
  case_00069        0.6258        -0.0034
  case_00176        0.6264        -0.0034
  case_00117        0.6299        -0.0033
  case_00138        0.6653        -0.0025
  case_00101        0.7018        -0.0016
  case_00098        0.7247        -0.0010
  case_00128        0.7405        -0.0007
  case_00180        0.7670        -0.0000

...
  case_00079        0.9500
  case_00024        0.9597
  case_00026        0.9622
  case_00025        0.9639
  case_00135        0.9716

SIMULATION: What if fewer hard cases in val?
  All 42 cases:              0.7672
  Remove 1 worst:            0.

In [16]:
import numpy as np
from pathlib import Path

ROI_NNUNET = Path(r"D:\fsdf_uca_kits\outputs\kidney_rois_nnunet_80_20")

zero_cases = ['case_00062', 'case_00106', 'case_00129', 'case_00137']

for case in zero_cases:
    rois = sorted(ROI_NNUNET.glob(f"{case}_roi*_seg.npy"))
    print(f"\n{case}: {len(rois)} ROIs")
    total_tumor = 0
    for roi_path in rois:
        seg = np.load(roi_path)
        roi_name = roi_path.stem.replace('_seg', '')
        tumor_vox = (seg == 2).sum()
        kidney_vox = (seg >= 1).sum()
        total_tumor += tumor_vox
        print(f"  {roi_name}: shape={seg.shape}, tumor={tumor_vox}, kidney={kidney_vox}")
    
    if total_tumor == 0:
        print(f"  >>> NO TUMOR IN ANY ROI — Stage 1 missed it!")
    else:
        print(f"  >>> Tumor EXISTS ({total_tumor} voxels) — Stage 2 failed to detect it")



case_00062: 2 ROIs
  case_00062_roi0: shape=(41, 148, 134), tumor=2638, kidney=134356
  case_00062_roi1: shape=(42, 135, 134), tumor=0, kidney=135909
  >>> Tumor EXISTS (2638 voxels) — Stage 2 failed to detect it

case_00106: 2 ROIs
  case_00106_roi0: shape=(46, 119, 131), tumor=0, kidney=141485
  case_00106_roi1: shape=(45, 126, 139), tumor=1744, kidney=153678
  >>> Tumor EXISTS (1744 voxels) — Stage 2 failed to detect it

case_00129: 2 ROIs
  case_00129_roi0: shape=(43, 120, 131), tumor=3506, kidney=121228
  case_00129_roi1: shape=(42, 123, 118), tumor=0, kidney=104662
  >>> Tumor EXISTS (3506 voxels) — Stage 2 failed to detect it

case_00137: 2 ROIs
  case_00137_roi0: shape=(46, 115, 111), tumor=0, kidney=94941
  case_00137_roi1: shape=(42, 119, 116), tumor=1143, kidney=87959
  >>> Tumor EXISTS (1143 voxels) — Stage 2 failed to detect it


In [17]:
import torch
import numpy as np
from pathlib import Path

ROI_NNUNET = Path(r"D:\fsdf_uca_kits\outputs\kidney_rois_nnunet_80_20")

# Load model (already loaded from eval)
model.eval()

zero_case_names = ['case_00062', 'case_00106', 'case_00129', 'case_00137']
zero_cases = []
for cn in zero_case_names:
    for seg_path in sorted(ROI_NNUNET.glob(f'{cn}_roi*_seg.npy')):
        seg_tmp = np.load(seg_path)
        if (seg_tmp == 2).any():
            rn = seg_path.stem.replace('_seg', '')
            zero_cases.append((cn, rn))
            break

for case_name, roi_name in zero_cases:
    img = np.load(ROI_NNUNET / f'{roi_name}_img.npy')
    seg = np.load(ROI_NNUNET / f'{roi_name}_seg.npy')
    D, H, W = img.shape
    half = SLICE_CONTEXT // 2
    
    max_dim = max(H, W)
    pad_h = (max_dim - H) // 2
    pad_w = (max_dim - W) // 2
    pad_val = float(img.min())
    
    # Find slices WITH tumor
    tumor_slices = [s for s in range(D) if (seg[s] == 2).any()]
    
    max_tumor_prob = 0.0
    max_tumor_slice = -1
    
    with torch.no_grad():
        for s in tumor_slices:
            slices = np.zeros((SLICE_CONTEXT, H, W), dtype=np.float32)
            for j in range(SLICE_CONTEXT):
                si = max(0, min(D-1, s - half + j))
                slices[j] = img[si]
            
            sq = np.full((SLICE_CONTEXT, max_dim, max_dim), pad_val, dtype=np.float32)
            sq[:, pad_h:pad_h+H, pad_w:pad_w+W] = slices
            
            inp = torch.nn.functional.interpolate(
                torch.from_numpy(sq[None]).float().to(device),
                (IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
            
            logits = model(inp)["logits"]  # V6 always returns tuple
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            
            # Tumor probability (class 2) 
            tumor_prob_max = probs[2].max()
            tumor_prob_mean = probs[2].mean()
            
            if tumor_prob_max > max_tumor_prob:
                max_tumor_prob = tumor_prob_max
                max_tumor_slice = s
    
    tumor_vox = (seg == 2).sum()
    print(f"{case_name} ({roi_name}):")
    print(f"  Tumor voxels: {tumor_vox}")
    print(f"  Tumor slices: {len(tumor_slices)}")
    print(f"  Max tumor PROBABILITY on any tumor slice: {max_tumor_prob:.4f}")
    print(f"  Best slice: {max_tumor_slice}")
    print(f"  {'→ Model SEES tumor but smoothing kills it' if max_tumor_prob > 0.3 else ''}")
    print(f"  {'→ Model NEVER detects tumor (true miss)' if max_tumor_prob < 0.1 else ''}")
    print()

case_00062 (case_00062_roi0):
  Tumor voxels: 2638
  Tumor slices: 8
  Max tumor PROBABILITY on any tumor slice: 1.0000
  Best slice: 23
  → Model SEES tumor but smoothing kills it
  

case_00106 (case_00106_roi1):
  Tumor voxels: 1744
  Tumor slices: 6
  Max tumor PROBABILITY on any tumor slice: 1.0000
  Best slice: 22
  → Model SEES tumor but smoothing kills it
  

case_00129 (case_00129_roi0):
  Tumor voxels: 3506
  Tumor slices: 7
  Max tumor PROBABILITY on any tumor slice: 1.0000
  Best slice: 34
  → Model SEES tumor but smoothing kills it
  

case_00137 (case_00137_roi1):
  Tumor voxels: 1143
  Tumor slices: 6
  Max tumor PROBABILITY on any tumor slice: 1.0000
  Best slice: 5
  → Model SEES tumor but smoothing kills it
  



In [18]:
import numpy as np
import json, time
from pathlib import Path
from scipy.ndimage import gaussian_filter

ROI_NNUNET_DIR = OUTPUT_DIR / "kidney_rois_nnunet_80_20"
FOLD_PATH = Path(r"D:\fsdf_uca_kits\outputs\stats\fold_splits_80_20.json")

model = FSDFUCANetV6(SLICE_CONTEXT, NUM_CLASSES, FUSE_CHANNELS, ENCODER).to(device)
ckpt = torch.load(CKPT_DIR / 'nb07e_fold0_best.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Loaded: epoch {ckpt['epoch']}")

with open(FOLD_PATH) as f:
    splits = json.load(f)
val_case_names = set(splits['folds']['fold_0']['val'])

with open(ROI_NNUNET_DIR / "roi_meta_all.json") as f:
    nn_metas = json.load(f)
nn_val = [m for m in nn_metas if m['case_name'] in val_case_names]
cases_tumor = sorted(set(m['case_name'] for m in nn_val if m['has_tumor']))

def infer_roi_multi_smooth(model, roi_name, roi_dir, img_size=256, n_slices=7, device='cuda'):
    img = np.load(roi_dir / f'{roi_name}_img.npy')
    seg = np.load(roi_dir / f'{roi_name}_seg.npy')
    D, H, W = img.shape
    half = n_slices // 2
    prob_vol = np.zeros((3, D, H, W), dtype=np.float32)
    
    max_dim = max(H, W)
    pad_h = (max_dim - H) // 2
    pad_w = (max_dim - W) // 2
    pad_val = float(img.min())
    
    model.eval()
    with torch.no_grad():
        for s in range(D):
            slices = np.zeros((n_slices, H, W), dtype=np.float32)
            for j in range(n_slices):
                si = max(0, min(D-1, s - half + j))
                slices[j] = img[si]
            
            sq = np.full((n_slices, max_dim, max_dim), pad_val, dtype=np.float32)
            sq[:, pad_h:pad_h+H, pad_w:pad_w+W] = slices
            
            # TTA: 4 flips
            probs_sum = np.zeros((3, img_size, img_size), dtype=np.float32)
            for flip_lr, flip_ud in [(False,False),(True,False),(False,True),(True,True)]:
                sq_aug = sq.copy()
                if flip_lr: sq_aug = sq_aug[:, :, ::-1].copy()
                if flip_ud: sq_aug = sq_aug[:, ::-1, :].copy()
                
                inp = torch.nn.functional.interpolate(
                    torch.from_numpy(sq_aug[None]).float().to(device),
                    (img_size, img_size), mode='bilinear', align_corners=False)
                logits = model(inp)["logits"]
                if isinstance(logits, tuple): logits = logits[0]
                p = torch.softmax(logits, dim=1).cpu().numpy()[0]
                
                if flip_lr: p = p[:, :, ::-1].copy()
                if flip_ud: p = p[:, ::-1, :].copy()
                probs_sum += p
            
            probs_avg = probs_sum / 4.0
            probs_full = np.zeros((3, max_dim, max_dim), dtype=np.float32)
            for c in range(3):
                probs_full[c] = torch.nn.functional.interpolate(
                    torch.from_numpy(probs_avg[c:c+1, None]),
                    (max_dim, max_dim), mode='bilinear', align_corners=False
                ).numpy()[0, 0]
            prob_vol[:, s] = probs_full[:, pad_h:pad_h+H, pad_w:pad_w+W]
    
    return prob_vol, seg

# Test 4 smoothing strategies
strategies = {
    'no_smooth': None,
    'light': [0.5, 0.3, 0.3],
    'current': [1.0, 0.5, 0.5],
    'kidney_only': 'kidney_only',  # smooth kidney, not tumor
}

all_results = {}

for strat_name, sigma in strategies.items():
    results = []
    t0 = time.time()
    
    for ci, case_name in enumerate(cases_tumor):
        pred_rois = sorted(ROI_NNUNET_DIR.glob(f"{case_name}_roi*_img.npy"))
        tp_m, fp_m, fn_m, tp_t, fp_t, fn_t = 0,0,0,0,0,0
        
        for p in pred_rois:
            roi_name = p.stem.replace('_img', '')
            if not (ROI_NNUNET_DIR / f'{roi_name}_seg.npy').exists(): continue
            
            prob_vol, gt_vol = infer_roi_multi_smooth(
                model, roi_name, ROI_NNUNET_DIR,
                img_size=IMG_SIZE, n_slices=SLICE_CONTEXT, device=device)
            
            # Apply smoothing strategy
            if sigma is None:
                pass  # no smoothing
            elif sigma == 'kidney_only':
                # Smooth only kidney+background, leave tumor raw
                prob_vol[0] = gaussian_filter(prob_vol[0], sigma=[1.0, 0.5, 0.5])  # bg
                prob_vol[1] = gaussian_filter(prob_vol[1], sigma=[1.0, 0.5, 0.5])  # kidney
                # prob_vol[2] NOT smoothed (tumor preserved)
            else:
                for c in range(3):
                    prob_vol[c] = gaussian_filter(prob_vol[c], sigma=sigma)
            
            pred_vol = prob_vol.argmax(axis=0).astype(np.uint8)
            
            pt = (pred_vol == 2); gt = (gt_vol == 2)
            tp_t += (pt & gt).sum(); fp_t += (pt & ~gt).sum(); fn_t += (~pt & gt).sum()
            pm = (pred_vol >= 1); gm = (gt_vol >= 1)
            tp_m += (pm & gm).sum(); fp_m += (pm & ~gm).sum(); fn_m += (~pm & gm).sum()
            del prob_vol, pred_vol, gt_vol
        
        torch.cuda.empty_cache()
        kid = float(2*tp_m / max(2*tp_m+fp_m+fn_m, 1))
        tum = float(2*tp_t / max(2*tp_t+fp_t+fn_t, 1))
        results.append({'case_name': case_name, 'kidney_merged': kid, 'tumor_dice': tum})
    
    all_results[strat_name] = results
    ek = np.mean([r['kidney_merged'] for r in results])
    et = np.mean([r['tumor_dice'] for r in results])
    print(f"{strat_name:>15}: Kidney={ek:.4f}, Tumor={et:.4f} ({time.time()-t0:.0f}s)")
    
    # Show the 4 problem cases
    for r in results:
        if r['case_name'] in ['case_00062', 'case_00106', 'case_00129', 'case_00137']:
            print(f"    {r['case_name']}: TumD={r['tumor_dice']:.4f}")

print(f"\n{'='*60}")
print(f"SMOOTHING COMPARISON")
print(f"{'='*60}")
print(f"{'Strategy':>15} {'Kidney':>10} {'Tumor':>10}")
for strat_name in strategies:
    ek = np.mean([r['kidney_merged'] for r in all_results[strat_name]])
    et = np.mean([r['tumor_dice'] for r in all_results[strat_name]])
    print(f"{strat_name:>15} {ek:>10.4f} {et:>10.4f}")

  TumorAwareHead bottleneck: 448 channels
Loaded: epoch 60
      no_smooth: Kidney=0.9632, Tumor=0.7629 (793s)
          light: Kidney=0.9633, Tumor=0.7644 (800s)
        current: Kidney=0.9623, Tumor=0.7672 (822s)


KeyboardInterrupt: 

In [ ]:

# Quick test: just the 4 zero-score cases with different smoothing
import numpy as np
from pathlib import Path
from scipy.ndimage import gaussian_filter

ROI_NNUNET_DIR = OUTPUT_DIR / "kidney_rois_nnunet_80_20"

model.eval()
test_cases = ['case_00062', 'case_00106', 'case_00129', 'case_00137']

for case_name in test_cases:
    roi_path = sorted(ROI_NNUNET_DIR.glob(f"{case_name}_roi*_img.npy"))
    
    for p in roi_path:
        roi_name = p.stem.replace('_img', '')
        seg = np.load(ROI_NNUNET_DIR / f'{roi_name}_seg.npy')
        if (seg == 2).sum() == 0:
            continue  # skip non-tumor ROIs
        
        img = np.load(ROI_NNUNET_DIR / f'{roi_name}_img.npy')
        D, H, W = img.shape
        half = SLICE_CONTEXT // 2
        prob_vol = np.zeros((3, D, H, W), dtype=np.float32)
        
        max_dim = max(H, W)
        pad_h = (max_dim - H) // 2
        pad_w = (max_dim - W) // 2
        pad_val = float(img.min())
        
        with torch.no_grad():
            for s in range(D):
                slices = np.zeros((SLICE_CONTEXT, H, W), dtype=np.float32)
                for j in range(SLICE_CONTEXT):
                    si = max(0, min(D-1, s - half + j))
                    slices[j] = img[si]
                
                sq = np.full((SLICE_CONTEXT, max_dim, max_dim), pad_val, dtype=np.float32)
                sq[:, pad_h:pad_h+H, pad_w:pad_w+W] = slices
                
                inp = torch.nn.functional.interpolate(
                    torch.from_numpy(sq[None]).float().to(device),
                    (IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
                logits = model(inp)["logits"]
                if isinstance(logits, tuple): logits = logits[0]
                probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
                
                probs_full = np.zeros((3, max_dim, max_dim), dtype=np.float32)
                for c in range(3):
                    probs_full[c] = torch.nn.functional.interpolate(
                        torch.from_numpy(probs[c:c+1, None]),
                        (max_dim, max_dim), mode='bilinear', align_corners=False
                    ).numpy()[0, 0]
                prob_vol[:, s] = probs_full[:, pad_h:pad_h+H, pad_w:pad_w+W]
        
        gt = seg
        
        # Test different smoothing
        print(f"\n{roi_name}: tumor={int((gt==2).sum())} voxels")
        
        for name, sigma in [('no_smooth', None), ('light [0.5,0.3,0.3]', [0.5,0.3,0.3]), 
                             ('current [1.0,0.5,0.5]', [1.0,0.5,0.5]),
                             ('kidney_only', 'kidney_only')]:
            pv = prob_vol.copy()
            if sigma is None:
                pass
            elif sigma == 'kidney_only':
                pv[0] = gaussian_filter(pv[0], sigma=[1.0, 0.5, 0.5])
                pv[1] = gaussian_filter(pv[1], sigma=[1.0, 0.5, 0.5])
            else:
                for c in range(3):
                    pv[c] = gaussian_filter(pv[c], sigma=sigma)
            
            pred = pv.argmax(axis=0).astype(np.uint8)
            pt = (pred == 2); gtt = (gt == 2)
            tp = (pt & gtt).sum(); fp = (pt & ~gtt).sum(); fn = (~pt & gtt).sum()
            dice = float(2*tp / max(2*tp+fp+fn, 1))
            pred_tumor = (pred == 2).sum()
            print(f"  {name:>25}: TumD={dice:.4f} (pred_tumor={pred_tumor})")


In [ ]:
import json
from pathlib import Path

OUTPUT_DIR = Path(r"D:\fsdf_uca_kits\outputs")
test_cases = ['case_00062', 'case_00106', 'case_00129', 'case_00137']

# Load old Stage 1 E2E results (NB04d on SwinUNETR ROIs)
old_e2e_path = OUTPUT_DIR / 'results' / 'fold0_nb04d_e2e_results.json'
if old_e2e_path.exists():
    with open(old_e2e_path) as f:
        old_results = {r['case_name']: r['tumor_dice'] for r in json.load(f)}
else:
    old_results = {}

# nnU-Net E2E results
nn_path = OUTPUT_DIR / 'results' / 'fold0_nb07e_e2e.json'
with open(nn_path) as f:
    nn_results = {r['case_name']: r['tumor_dice'] for r in json.load(f)}

print(f"{'Case':<15} {'Old Stage1':>12} {'nnU-Net Stage1':>15} {'Change':>10}")
print("-" * 55)
for case in test_cases:
    old = old_results.get(case, -1)
    new = nn_results.get(case, -1)
    print(f"  {case:<15} {old:>12.4f} {new:>15.4f} {new-old:>+10.4f}")

# Check ROI geometry
import numpy as np
ROI_OLD = Path(r"D:\fsdf_uca_kits\outputs\kidney_rois_pred")
ROI_NN = Path(r"D:\fsdf_uca_kits\outputs\kidney_rois_nnunet_80_20")

print(f"\nROI COMPARISON:")
for case in test_cases:
    print(f"\n  {case}:")
    for src, roi_dir in [("OldPred", ROI_OLD), ("nnUNet", ROI_NN)]:
        for seg_path in sorted(roi_dir.glob(f"{case}_roi*_seg.npy")):
            roi_name = seg_path.stem.replace('_seg', '')
            seg = np.load(seg_path)
            img = np.load(roi_dir / f'{roi_name}_img.npy')
            tumor = (seg == 2).sum()
            kidney = (seg >= 1).sum()
            if tumor > 0:
                print(f"    {src} {roi_name}: shape={img.shape}, tumor={tumor}, kidney={kidney}, tumor%={100*tumor/seg.size:.1f}%")

In [ ]:
import numpy as np
import json
from pathlib import Path

with open(Path(r"D:\fsdf_uca_kits\outputs\results\fold0_nb07e_tta_pp.json")) as f:
    data = json.load(f)
results = data['tta_pp']

scores = [(r['case_name'], r['tumor_dice']) for r in results]
scores_sorted = sorted(scores, key=lambda x: x[1])

hard = [(n,s) for n,s in scores_sorted if s < 0.40]
medium = [(n,s) for n,s in scores_sorted if 0.40 <= s < 0.80]
easy = [(n,s) for n,s in scores_sorted if s >= 0.80]

print(f"Current 42-case distribution:")
print(f"  Hard:   {len(hard)} cases, mean={np.mean([s for _,s in hard]):.4f}")
print(f"  Medium: {len(medium)} cases, mean={np.mean([s for _,s in medium]):.4f}")
print(f"  Easy:   {len(easy)} cases, mean={np.mean([s for _,s in easy]):.4f}")
print(f"  Overall: {np.mean([s for _,s in scores]):.4f}")

np.random.seed(42)
strat_means = []

for trial in range(1000):
    n_hard = min(len(hard), max(1, int(42 * len(hard) / 70)))
    n_medium = min(len(medium), int(42 * len(medium) / 70))
    n_easy = 42 - n_hard - n_medium
    n_easy = min(len(easy), n_easy)
    
    h_idx = np.random.choice(len(hard), n_hard, replace=False)
    m_idx = np.random.choice(len(medium), n_medium, replace=False)
    e_idx = np.random.choice(len(easy), n_easy, replace=False)
    
    sample = [hard[i][1] for i in h_idx] + [medium[i][1] for i in m_idx] + [easy[i][1] for i in e_idx]
    strat_means.append(np.mean(sample))

improvement = 0.025
unfixable = {'case_00062', 'case_00106', 'case_00129', 'case_00137'}
fixable_scores = [s for n,s in scores if n not in unfixable]

print(f"\n{'='*60}")
print(f"STRATIFIED 42-CASE SPLIT SIMULATIONS")
print(f"{'='*60}")
print(f"  Stratified 42-case (1000 trials, SAME model):")
print(f"    Mean:  {np.mean(strat_means):.4f}")
print(f"    Std:   {np.std(strat_means):.4f}")
print(f"    Best:  {np.max(strat_means):.4f}")
print(f"    Worst: {np.min(strat_means):.4f}")
print(f"\n  With model improvement (+{improvement} from 30 more train cases):")
print(f"    Expected mean:  {np.mean(strat_means) + improvement:.4f}")
print(f"    Expected range: {np.min(strat_means) + improvement:.4f} - {np.max(strat_means) + improvement:.4f}")
print(f"\n  Without 4 unfixable cases ({len(fixable_scores)} cases):")
print(f"    Mean: {np.mean(fixable_scores):.4f}")

print(f"\n{'='*60}")
print(f"REALISTIC EXPECTED RESULTS WITH 80/20 SPLIT")
print(f"{'='*60}")
print(f"  Current (70 val):           0.709")
print(f"  Stratified 42 (same model): ~{np.mean(strat_means):.3f}")
print(f"  + model improvement:        ~{np.mean(strat_means)+improvement:.3f}")
print(f"  Best case:                  ~{np.max(strat_means)+improvement:.3f}")
print(f"  Worst case:                 ~{np.min(strat_means)+improvement:.3f}")
print(f"")
print(f"  MFFAU-Net reported:         0.875 (42-case self-split)")
print(f"  Santini official:           0.740 (90-case test)")